
# Medical Document Best Extraction Workbench v12.1 Hotfix

This is a small but important hotfix over v12.

The v12 results showed one critical blocker:

- clean TAV text-PDF reports had valid patient/date/common fields and safe rows, but were wrongly classified as `continuation_or_body_only_page`;
- therefore all 7 clean PDFs became `manual_review`;
- `safe_payload_candidates.jsonl` became empty.

v12.1 fixes page-context logic so a document is considered standalone when extracted/validated common fields prove patient identity + report date, even if the raw OCR text does not contain labels like `Patient Name`.

Expected after rerun:
- 7 clean PDFs should return `recommended_save=True`;
- `safe_payload_candidates.jsonl` should contain 7 payloads;
- image reports remain review/reupload unless both document context and row validation pass.


In [1]:
from pathlib import Path
import os, re, json, html, hashlib, traceback, subprocess, unicodedata
from dataclasses import dataclass, field, asdict
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from PIL import Image, ImageOps, ImageEnhance, ImageFilter

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False

try:
    import fitz
    PYMUPDF_AVAILABLE = True
except Exception:
    fitz = None
    PYMUPDF_AVAILABLE = False

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    TESSERACT_AVAILABLE = False

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "samples").exists() or (p / "app").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
SELECTED_DIR = REPO_ROOT / "samples" / "selected samples"
OUTPUT_DIR = REPO_ROOT / "notebook_outputs" / "selected_samples_v12_1_hotfix"
TEXT_DIR = OUTPUT_DIR / "chosen_texts"
SANITIZED_TEXT_DIR = OUTPUT_DIR / "sanitized_texts"
JSON_DIR = OUTPUT_DIR / "json"
PREVIEW_DIR = OUTPUT_DIR / "source_previews"
DEBUG_DIR = OUTPUT_DIR / "debug"
VARIANT_IMAGE_DIR = DEBUG_DIR / "variant_images"
VARIANT_TEXT_DIR = DEBUG_DIR / "variant_texts"

for d in [OUTPUT_DIR, TEXT_DIR, SANITIZED_TEXT_DIR, JSON_DIR, PREVIEW_DIR, DEBUG_DIR, VARIANT_IMAGE_DIR, VARIANT_TEXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".pdf", ".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}

def relpath(p):
    try:
        return str(Path(p).resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(p)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Tesseract:", TESSERACT_AVAILABLE, "| OpenCV:", CV2_AVAILABLE, "| PyMuPDF:", PYMUPDF_AVAILABLE)


Repo root: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents
Output dir: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/notebook_outputs/selected_samples_v12_1_hotfix
Tesseract: True | OpenCV: True | PyMuPDF: True


In [2]:
# =========================
# Git hygiene
# =========================

AUTO_UPDATE_GITIGNORE = True

RECOMMENDED_GITIGNORE_LINES = [
    "samples/selected samples/",
    "samples/selected_samples/",
    "notebook_outputs/",
    "debug_output/",
    "eval_storage/",
    "*.db",
]

def run_git(args):
    try:
        return subprocess.run(["git"] + args, cwd=REPO_ROOT, capture_output=True, text=True, check=False)
    except Exception:
        return None

def update_gitignore():
    gi = REPO_ROOT / ".gitignore"
    existing = gi.read_text(encoding="utf-8") if gi.exists() else ""
    missing_before = [x for x in RECOMMENDED_GITIGNORE_LINES if x not in existing]
    if AUTO_UPDATE_GITIGNORE and missing_before:
        with gi.open("a", encoding="utf-8") as f:
            f.write("\n# Medical extraction PHI/PII and notebook outputs\n")
            for x in missing_before:
                f.write(x + "\n")
    existing_after = gi.read_text(encoding="utf-8") if gi.exists() else ""
    missing_after = [x for x in RECOMMENDED_GITIGNORE_LINES if x not in existing_after]
    tracked = []
    res = run_git(["ls-files"])
    if res and res.returncode == 0:
        for x in res.stdout.splitlines():
            if x.startswith("samples/selected samples/") or x.startswith("samples/selected_samples/") or x.startswith("notebook_outputs/"):
                tracked.append(x)
    report = {
        "gitignore_path": relpath(gi),
        "missing_before": missing_before,
        "missing_after": missing_after,
        "tracked_sensitive_files_count": len(tracked),
        "tracked_sensitive_files_sample": tracked[:50],
        "cleanup_commands": [
            "git rm -r --cached 'samples/selected samples' 'samples/selected_samples' 'notebook_outputs' 2>/dev/null || true",
            "git add .gitignore",
            "git commit -m 'chore: ignore PHI samples and notebook outputs'",
        ],
    }
    (OUTPUT_DIR / "privacy_git_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

print(json.dumps(update_gitignore(), ensure_ascii=False, indent=2))


{
  "gitignore_path": ".gitignore",
  "missing_before": [],
  "missing_after": [],
  "tracked_sensitive_files_count": 1513,
  "tracked_sensitive_files_sample": [
    "notebook_outputs/selected_samples_flow/json/0001_20260427_181919.json",
    "notebook_outputs/selected_samples_flow/json/0003_0014161672_14041209_O_00404121721.json",
    "notebook_outputs/selected_samples_flow/json/0005_20260427_181713.json",
    "notebook_outputs/selected_samples_flow/json/0006_0021858845_14041209_O_00404121731.json",
    "notebook_outputs/selected_samples_flow/json/0007_-2147483648_-210195.json",
    "notebook_outputs/selected_samples_flow/json/0008_0023471026_14041209_O_00404121728.json",
    "notebook_outputs/selected_samples_flow/json/0009_0025631314_14041209_O_00404121726.json",
    "notebook_outputs/selected_samples_flow/json/0010_-2147483648_-210193.json",
    "notebook_outputs/selected_samples_flow/json/0012_0024150010_14041209_O_00404121722.json",
    "notebook_outputs/selected_samples_flow/jso

In [3]:
# =========================
# Discover files
# =========================

ONLY_FILES = []  # optional: ["samples/selected samples/file.jpg"]

def discover_files():
    if ONLY_FILES:
        files = []
        for x in ONLY_FILES:
            p = Path(x)
            if not p.is_absolute():
                p = REPO_ROOT / p
            if p.exists() and p.is_file() and p.suffix.lower() in SUPPORTED_EXTS:
                files.append(p.resolve())
        return files

    folders = [SELECTED_DIR]
    found = []
    for folder in folders:
        if folder.exists():
            for ext in SUPPORTED_EXTS:
                found.extend(folder.rglob(f"*{ext}"))
                found.extend(folder.rglob(f"*{ext.upper()}"))
    seen, clean = set(), []
    for p in found:
        rp = p.resolve()
        if rp.is_file() and rp not in seen:
            seen.add(rp)
            clean.append(rp)
    return sorted(clean, key=lambda p: p.name)

FILES = discover_files()
print("Found", len(FILES), "files")
for i, p in enumerate(FILES, 1):
    print(f"{i:03d} | {p.name} | {p.suffix.lower()} | {p.stat().st_size/1024:.1f} KB | {relpath(p)}")

assert FILES, "No files found. Put files in samples/selected samples/ or set ONLY_FILES."


Found 20 files
001 | -2147483648_-210193.jpg | .jpg | 77.8 KB | samples/selected samples/-2147483648_-210193.jpg
002 | -2147483648_-210195.jpg | .jpg | 42.0 KB | samples/selected samples/-2147483648_-210195.jpg
003 | 0014161672_14041209_O_00404121721.pdf | .pdf | 44.2 KB | samples/selected samples/0014161672_14041209_O_00404121721.pdf
004 | 0020139871_14041209_O_00404121714.pdf | .pdf | 41.4 KB | samples/selected samples/0020139871_14041209_O_00404121714.pdf
005 | 0021858845_14041209_O_00404121731.pdf | .pdf | 42.9 KB | samples/selected samples/0021858845_14041209_O_00404121731.pdf
006 | 0023471026_14041209_O_00404121728.pdf | .pdf | 40.8 KB | samples/selected samples/0023471026_14041209_O_00404121728.pdf
007 | 0024150010_14041209_O_00404121722.pdf | .pdf | 44.1 KB | samples/selected samples/0024150010_14041209_O_00404121722.pdf
008 | 0025631314_14041209_O_00404121726.pdf | .pdf | 41.2 KB | samples/selected samples/0025631314_14041209_O_00404121726.pdf
009 | 0025692283_14041209_O_00404

In [4]:
# =========================
# Config, aliases, normalization
# =========================

CONFIG = {
    "max_pdf_pages": 5,
    "min_ocr_text_len": 20,
    "psm_modes": [4, 6, 11, 12],
    "tesseract_langs": ["eng+fas", "eng"],
    "run_rotation_variants": True,
    "save_variant_images": True,
    "save_variant_texts": True,
    "min_good_layout_score": 0.62,
    "min_usable_layout_score": 0.38,
    "backend_patient_context_available": False,
    "require_report_date_for_backend_save": True,
    "require_patient_or_backend_context_for_backend_save": True,
}

MEDICAL_KEYWORDS = [
    "laboratory","lab","hematology","biochemistry","serology","hormone","thyroid","urine","culture",
    "cbc","wbc","rbc","hemoglobin","platelet","fbs","glucose","creatinine","cholesterol","triglycerides",
    "tsh","ferritin","vitamin","crp","pt","inr","ptt","reference range","method",
    "آزمایشگاه","آزمايشگاه","پاتوبیولوژی","پاتوبيولوژی","درمانگاه","بیمارستان","پذیرش","نتیجه","ادرار","خون","نام بیمار",
]

LAB_ALIASES = {
    "WBC": ["WBC","W.B.C","White Blood Cell"],
    "RBC": ["RBC","R.B.C","Red Blood Cell"],
    "HGB": ["HGB","Hb","Hemoglobin","Haemoglobin","Hemoglobine"],
    "HCT": ["HCT","H.C.T","Hematocrit","Het"],
    "MCV": ["MCV","M.C.V"],
    "MCH": ["MCH","M.C.H"],
    "MCHC": ["MCHC","M.C.H.C"],
    "RDW-CV": ["RDW-CV","RDW CV","RDW."],
    "RDW-SD": ["RDW-SD","RDW SD"],
    "PLT": ["PLT","Platelet","Platelets"],
    "PDW": ["PDW"],
    "MPV": ["MPV"],
    "NEUT%": ["NEUT%","Neut%","Neutrophils","Neutrophil"],
    "LYMPH%": ["LYMPH%","Lymph%","Lymphocytes","Lymphocyte","Lympho"],
    "MONO%": ["MONO%","Mono%","Monocytes","Monocyte"],
    "EO%": ["EO%","EOS%","Eosinophils","Eosinophil"],
    "BASO%": ["BASO%","Basophils","Basophil"],
    "ESR": ["ESR","Erythrocyte Sediment Rate","Erythrocyte Sedimentation Rate"],
    "FBS": ["Fasting Blood Glucose","Fasting blood sugar","Fasting blood segar","FBS","Glucose","Blood sugar"],
    "HbA1c": ["HbA1c","Hemoglobin A1c","Hemoglobine A1c","Hemoglobin Ate"],
    "EAG": ["Estimated Average Glucose","EAG","EAS"],
    "BUN": ["BUN","Blood Urea Nitrogen","Blood Urea nitrogen"],
    "Urea": ["Urea"],
    "Creatinine": ["Creatinine","Creatininee","Creatinin"],
    "Uric Acid": ["Uric Acid"],
    "Total Cholesterol": ["Total Cholesterol","Cholesterol","Cholestrol"],
    "Triglycerides": ["Triglycerides","Triglycerideses","Triglycerid","TG"],
    "HDL": ["HDL","HDL Cholesterol","HDL-Cholesterol"],
    "LDL": ["LDL","LDL Cholesterol","LDL-Cholesterol","LDL - Cholesterol"],
    "AST": ["AST","SGOT","SGOT(AST)","AST(SGOT)"],
    "ALT": ["ALT","SGPT","SGPT(ALT)"],
    "TSH": ["TSH","Thyroid Stimulating Hormone"],
    "T3": ["T3","Tri iodothyronine","Triiodothyronine"],
    "T4": ["T4","Thyroxine"],
    "Free T4": ["Free T4","FT4"],
    "Vitamin D": ["Vitamin D","25 Hydroxy Vitamin D","25-Hydroxy Vitamin D3","25-OH Vitamin D3"],
    "Ferritin": ["Ferritin","Ferrin"],
    "CRP": ["CRP","C-Reactive protein"],
    "PT": ["PT","Prothrombin Time"],
    "PT Control": ["PT Control","PT Controt"],
    "INR": ["INR"],
    "PTT": ["PTT","Activated PTT","Activated PT"],
    "Urine Culture": ["Urine Culture","Urine Cultures","Culture"],
    "Color": ["Color","olor"],
    "Appearance": ["Appearance","Appenrance"],
    "Specific Gravity": ["Specific Gravity","Sp. Gravity","Specie Gravity"],
    "pH": ["pH","PH"],
    "Protein": ["Protein"],
    "Urine Glucose": ["Urine Glucose","Glucose"],
    "Ketone": ["Ketone","Kewoe"],
    "Bilirubin Urine": ["Bilirubin","Darwin"],
    "Urobilinogen": ["Urobilinogen","Uobinogen"],
    "Nitrite": ["Nitrite","Nirive"],
    "Blood/Hb": ["Blood/Hb","Hemoglobin","Blood"],
    "WBC/HPF": ["WBC/HPF","W.B.C / HPF"],
    "RBC/HPF": ["RBC/HPF","R.B.C / HPF"],
    "Bacteria": ["Bacteria","Dacterta","acterta"],
    "Mucus": ["Mucus"],
    "Casts": ["Casts","Cast"],
    "Crystals": ["Crystals","Crystal"],
}

ALL_ALIAS_PHRASES = [(std, alias) for std, aliases in LAB_ALIASES.items() for alias in aliases]
ALL_ALIAS_LOWER = [(std, alias, alias.lower()) for std, alias in ALL_ALIAS_PHRASES]

ROW_SANITY_LIMITS = {
    "WBC": (0.5,80), "RBC": (1.0,8.0), "HGB": (3.0,25.0), "HCT": (10.0,75.0),
    "MCV": (50.0,130.0), "MCH": (15.0,45.0), "MCHC": (20.0,45.0), "PLT": (10.0,1500.0),
    "HbA1c": (3.0,20.0), "FBS": (20.0,600.0), "Creatinine": (0.1,20.0),
    "Ferritin": (1.0,3000.0), "Vitamin D": (1.0,200.0), "TSH": (0.001,200.0),
}

QUALITATIVE_VALID_VALUES = {
    "negative","positive","trace","few","rare","many","not seen","clear","yellow","amber","red","pale","straw","normal",
    "turbid","cloudy","semi-clear","acid","alkaline","look at culture","no growth after 24 hrs","no growth after 48 hrs",
    "no growth after 24 hours","no growth after 48 hours","no bacteria growth after 48 hrs",
}
QUALITATIVE_TESTS = {
    "Urine Culture","Color","Appearance","Protein","Urine Glucose","Ketone","Bilirubin Urine","Urobilinogen",
    "Nitrite","Blood/Hb","Bacteria","Mucus","Casts","Crystals",
}

DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")

def normalize_digits(text):
    return (text or "").translate(DIGIT_MAP)

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text or "")
    text = normalize_digits(text)
    text = text.replace("ي","ی").replace("ك","ک").replace("ۀ","ه")
    corrections = {
        "خائم":"خانم","خانمز":"خانم ز","خانمزهرا":"خانم زهرا","خانمسیده":"خانم سیده","خانمفرزانه":"خانم فرزانه",
        "بیمار نام":"نام بیمار","بيمار نام":"نام بیمار","پزشک نام":"نام پزشک","پزشك نام":"نام پزشک",
        "پذیرش تاریخ":"تاریخ پذیرش","پذيرش تاریخ":"تاریخ پذیرش","پدیرش":"پذیرش","پذبرش":"پذیرش",
        "پذيرش":"پذیرش","تاريخ":"تاریخ","دكتر":"دکتر","پزشك":"پزشک",
        "Fasting Blood Glucase":"Fasting Blood Glucose","Fasting blood segar":"Fasting blood sugar",
        "Hemoglobine Alc":"Hemoglobin A1c","Hemoglobin Ate":"Hemoglobin A1c","Cholestrol":"Cholesterol",
        "Triglycerideses":"Triglycerides","Creatininee":"Creatinine","Creatinin":"Creatinine",
    }
    for a,b in corrections.items():
        text = text.replace(a,b)
    text = re.sub(r"[^\S\n]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def clean_value(v, max_len=120):
    if not v:
        return None
    v = normalize_text(v)
    v = re.sub(r"^[\s:：|#*/\\.,؛;+_-]+|[\s:：|#*/\\.,؛;+_-]+$", "", v).strip()
    v = re.sub(r"\s+", " ", v)
    if not v or len(v) > max_len:
        return None
    return v

def sanitize_text(text):
    t = text or ""
    t = re.sub(r"(?<!\d)\d{10}(?!\d)", "**********", t)
    t = re.sub(r"(?<!\d)09\d{9}(?!\d)", "09*********", t)
    return t

def safe_id(path):
    stem = re.sub(r"[^\w\u0600-\u06FF.-]+", "_", Path(path).stem)
    return stem[:80] or hashlib.md5(str(path).encode()).hexdigest()[:10]

def hash_value(v):
    return hashlib.sha256(str(v).encode("utf-8")).hexdigest() if v else None

def validate_jalali_date(date):
    if not date:
        return False
    date = normalize_digits(date)
    m = re.match(r"^(13[9][0-9]|14[0-1][0-9]|1420)[/-](\d{1,2})[/-](\d{1,2})$", date)
    if not m:
        return False
    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return 1390 <= y <= 1420 and 1 <= mo <= 12 and 1 <= d <= 31

def file_mime_guess(path):
    return {".pdf":"application/pdf",".jpg":"image/jpeg",".jpeg":"image/jpeg",".png":"image/png",".webp":"image/webp",".tif":"image/tiff",".tiff":"image/tiff"}.get(Path(path).suffix.lower(), "application/octet-stream")


In [5]:
# =========================
# Image quality, preview, preprocessing variants
# =========================

def assess_image_quality(path):
    try:
        img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
        w,h = img.size
        gray = np.array(img.convert("L"))
        mean = float(np.mean(gray)); std = float(np.std(gray))
        lap = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if CV2_AVAILABLE else 0.0
        blur_score = min(1.0, lap/500.0) if lap else 0.5
        brightness_score = max(0.0, 1 - abs(mean-127)/127)
        contrast_score = min(1.0, std/80.0)
        resolution_score = min(1.0, (w*h)/(1000*1000))
        overall = blur_score*.35 + brightness_score*.20 + contrast_score*.25 + resolution_score*.20
        issues = []
        if lap and blur_score < .18: issues.append("severe_blur")
        if contrast_score < .25: issues.append("low_contrast")
        if mean < 45: issues.append("too_dark")
        if mean > 220: issues.append("too_bright")
        if resolution_score < .10: issues.append("low_resolution")
        if h > w*1.35 or w > h*1.35: issues.append("possible_rotation_or_sideways")
        status = "good_quality" if overall >= 0.60 else ("needs_preprocessing" if overall >= 0.22 else "poor_quality")
        return {"status":status,"overall_quality_score":round(overall,3),"issues":issues,"width":w,"height":h,
                "brightness_mean":round(mean,2),"contrast_std":round(std,2),"laplacian_variance":round(lap,2)}
    except Exception as e:
        return {"status":"unreadable","overall_quality_score":0.0,"issues":["unreadable_image",str(e)]}

def assess_file_quality(path):
    if Path(path).suffix.lower() == ".pdf":
        return {"status":"pdf_quality_not_assessed_here","overall_quality_score":None,"issues":[]}
    return assess_image_quality(path)

def create_source_preview(path, sid):
    out = PREVIEW_DIR / f"{sid}.png"
    try:
        if Path(path).suffix.lower() == ".pdf" and PYMUPDF_AVAILABLE:
            doc = fitz.open(path); page = doc[0]
            pix = page.get_pixmap(matrix=fitz.Matrix(2,2), alpha=False)
            pix.save(str(out)); doc.close()
        else:
            img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
            img.thumbnail((900,900)); img.save(out)
        return relpath(out)
    except Exception:
        return None

def pil_clahe(img, clip=2.0):
    if not CV2_AVAILABLE:
        return ImageOps.autocontrast(img.convert("L")).convert("RGB")
    gray = np.array(img.convert("L"))
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8,8))
    return Image.fromarray(clahe.apply(gray)).convert("RGB")

def pil_shadow_remove(img):
    if not CV2_AVAILABLE:
        return ImageOps.autocontrast(img.convert("L")).convert("RGB")
    arr = np.array(img.convert("RGB"))
    planes = cv2.split(arr)
    result_planes = []
    for plane in planes:
        dilated = cv2.dilate(plane, np.ones((7,7), np.uint8))
        bg = cv2.medianBlur(dilated, 21)
        diff = 255 - cv2.absdiff(plane, bg)
        norm = cv2.normalize(diff, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
        result_planes.append(norm)
    return Image.fromarray(cv2.merge(result_planes)).convert("RGB")

def image_preprocess_variants(path, sid):
    base = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    variants = []
    def add(name, im):
        variants.append((name, im.convert("RGB")))
    add("original_oriented", base)
    add("upscale2_original", base.resize((base.width*2, base.height*2), Image.Resampling.LANCZOS))
    if max(base.size) < 1500:
        add("upscale3_original", base.resize((base.width*3, base.height*3), Image.Resampling.LANCZOS))
    gray = ImageOps.grayscale(base)
    add("gray_autocontrast", ImageOps.autocontrast(gray))
    add("gray_autocontrast_upscale2", ImageOps.autocontrast(gray).resize((base.width*2, base.height*2), Image.Resampling.LANCZOS))
    add("mild_contrast", ImageEnhance.Contrast(gray).enhance(1.4))
    add("sharpen_light", ImageOps.autocontrast(gray.filter(ImageFilter.SHARPEN)))
    add("clahe_light", pil_clahe(base, 1.5))
    add("clahe_medium", pil_clahe(base, 2.5))
    add("shadow_removed", pil_shadow_remove(base))
    if CV2_AVAILABLE:
        arr = np.array(gray)
        thr = cv2.adaptiveThreshold(arr,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,11)
        add("threshold_fallback", Image.fromarray(thr))
    if CONFIG["run_rotation_variants"]:
        for name, im in list(variants):
            if name in {"original_oriented","gray_autocontrast","gray_autocontrast_upscale2","clahe_light","shadow_removed"}:
                for deg in [90,180,270]:
                    add(f"{name}_rot{deg}", im.rotate(deg, expand=True))
    if CONFIG["save_variant_images"]:
        folder = VARIANT_IMAGE_DIR / sid
        folder.mkdir(parents=True, exist_ok=True)
        for name, im in variants:
            im.save(folder / f"{name}.png")
    return variants


In [6]:
# =========================
# OCR word boxes, visual lines, scoring
# =========================

@dataclass
class OCRCandidate:
    candidate_id: str
    page_number: int
    variant_name: str
    psm: int | None
    lang: str
    text: str
    confidence: float
    score: float
    status: str
    score_details: dict
    words: list = field(default_factory=list)
    lines: list = field(default_factory=list)
    error: str | None = None

def run_tesseract_to_words(img, lang, psm):
    if not TESSERACT_AVAILABLE:
        return [], 0.0, "pytesseract_not_available"
    cfg = f"--psm {psm} -c preserve_interword_spaces=1"
    try:
        data = pytesseract.image_to_data(img, lang=lang, config=cfg, output_type=pytesseract.Output.DICT)
        words = []
        n = len(data.get("text", []))
        for i in range(n):
            txt = str(data["text"][i] or "").strip()
            if not txt:
                continue
            try:
                conf = float(data.get("conf", ["-1"])[i])
            except Exception:
                conf = -1
            if conf < 0:
                continue
            words.append({
                "text": normalize_text(txt),
                "conf": conf/100.0,
                "left": int(data.get("left",[0]*n)[i]),
                "top": int(data.get("top",[0]*n)[i]),
                "width": int(data.get("width",[0]*n)[i]),
                "height": int(data.get("height",[0]*n)[i]),
                "block_num": int(data.get("block_num",[0]*n)[i]),
                "par_num": int(data.get("par_num",[0]*n)[i]),
                "line_num": int(data.get("line_num",[0]*n)[i]),
                "word_num": int(data.get("word_num",[0]*n)[i]),
            })
        conf = float(np.mean([w["conf"] for w in words])) if words else 0.0
        return words, conf, None
    except Exception as e:
        return [], 0.0, str(e)

def reconstruct_visual_lines(words):
    if not words:
        return []
    groups = defaultdict(list)
    for w in words:
        groups[(w["block_num"], w["par_num"], w["line_num"])].append(w)
    lines = []
    for key, items in groups.items():
        items = sorted(items, key=lambda x: x["left"])
        text = " ".join(w["text"] for w in items).strip()
        if not text:
            continue
        left = min(w["left"] for w in items)
        top = min(w["top"] for w in items)
        right = max(w["left"] + w["width"] for w in items)
        bottom = max(w["top"] + w["height"] for w in items)
        lines.append({
            "line_id": 0, "text": normalize_text(text),
            "left": left, "top": top, "right": right, "bottom": bottom,
            "width": right-left, "height": bottom-top,
            "word_count": len(items),
            "avg_conf": float(np.mean([w["conf"] for w in items])),
        })
    lines = sorted(lines, key=lambda x: (x["top"], x["left"]))
    for i, line in enumerate(lines, 1):
        line["line_id"] = i
    return lines

def text_from_lines(lines):
    return "\n".join(l["text"] for l in lines if l.get("text"))

def alias_hits(text):
    low = normalize_text(text).lower()
    return [(std, alias) for std, alias, alow in ALL_ALIAS_LOWER if alow and alow in low]

def medical_hits(text):
    low = normalize_text(text).lower()
    return [k for k in MEDICAL_KEYWORDS if normalize_text(k).lower() in low]

def line_table_pattern_count(lines):
    count = 0
    for l in lines:
        txt = l["text"]
        has_alias = bool(alias_hits(txt))
        nums = re.findall(r"(?<!\d)[<>*]?\d+(?:\.\d+)?(?!\d)", txt)
        unit = re.search(r"(mg/dL|mg/dl|g/dL|U/L|%|fL|pg|10\^|ng/mL|uIU/mL)", txt, re.I)
        if has_alias and (nums or unit):
            count += 1
        elif len(nums) >= 2 and unit:
            count += 1
    return count

def common_field_signal_count(text):
    signals = [r"نام\s*بیمار", r"نام\s*مراجعه", r"کد\s*ملی", r"تاریخ\s*پذیرش", r"Report\s*Date", r"Patient\s*Name", r"سن", r"پذیرش"]
    return sum(1 for s in signals if re.search(s, text, re.I))

def impossible_value_penalty(text):
    penalties = 0
    for pat in [r"\bMCH\b[^\n]{0,40}\b\d{3,}\b", r"HbA1c[^\n]{0,40}\b\d{3,}\b", r"\bWBC\b[^\n]{0,40}\b[5-9]\d\b"]:
        if re.search(pat, text, re.I):
            penalties += 1
    return penalties

def gibberish_ratio(text):
    toks = re.findall(r"[A-Za-z\u0600-\u06FF0-9]+", text or "")
    if not toks:
        return 1.0
    random_short = sum(1 for t in toks if len(t) <= 2 and not t.isdigit())
    long_gib = sum(1 for t in toks if len(t) >= 12 and not re.search(r"[aeiouآایو]", t, re.I))
    return min(1.0, (random_short*0.25 + long_gib) / len(toks))

def culture_signal(text):
    return bool(re.search(r"No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*(?:hrs?|hours)\.?", normalize_text(text), re.I))

def score_ocr_candidate(text, lines, confidence):
    text = normalize_text(text)
    ah = alias_hits(text)
    mh = medical_hits(text)
    nums = re.findall(r"(?<!\d)[<>*]?\d+(?:\.\d+)?(?!\d)", text)
    table_patterns = line_table_pattern_count(lines)
    common_signals = common_field_signal_count(text)
    gib = gibberish_ratio(text)
    culture = culture_signal(text)
    impossible = impossible_value_penalty(text)
    line_count = len([l for l in lines if l.get("text")])
    avg_line_conf = float(np.mean([l["avg_conf"] for l in lines])) if lines else 0.0

    score = (
        min(len(text),3500)/3500*0.10 +
        min(len(ah),30)/30*0.23 +
        min(len(mh),18)/18*0.14 +
        min(len(nums),100)/100*0.10 +
        min(table_patterns,20)/20*0.18 +
        min(common_signals,8)/8*0.09 +
        min(line_count,60)/60*0.06 +
        max(confidence, avg_line_conf)*0.16 -
        gib*0.20 -
        min(impossible,5)/5*0.12
    )
    if culture:
        score = max(score, 0.42)
    score = max(0.0, min(1.0, float(score)))
    if len(text.strip()) < CONFIG["min_ocr_text_len"]:
        status = "poor_ocr_text"
    elif culture and len(text.strip()) >= 40:
        status = "usable_short_culture_text"
    elif score >= CONFIG["min_good_layout_score"]:
        status = "good_layout_text"
    elif score >= CONFIG["min_usable_layout_score"]:
        status = "usable_noisy_layout_text"
    else:
        status = "gibberish_or_bad_layout_text"
    details = {
        "alias_hits": len(ah),
        "medical_hits": len(mh),
        "numeric_hits": len(nums),
        "table_patterns": table_patterns,
        "common_field_signals": common_signals,
        "line_count": line_count,
        "gibberish_ratio": round(gib,3),
        "culture_signal": culture,
        "impossible_value_penalty": impossible,
        "avg_word_confidence": round(confidence,3),
        "avg_line_confidence": round(avg_line_conf,3),
    }
    return score, status, details

def ocr_image_candidate(img, page_number, variant_name, psm, lang, candidate_id):
    words, conf, err = run_tesseract_to_words(img, lang, psm)
    lines = reconstruct_visual_lines(words)
    text = text_from_lines(lines)
    score, status, details = score_ocr_candidate(text, lines, conf)
    return OCRCandidate(candidate_id, page_number, variant_name, psm, lang, text, conf, score, status, details, words, lines, err)


In [7]:
# =========================
# PDF/Image OCR candidates
# =========================

def extract_pdf_text_layer(path):
    if not PYMUPDF_AVAILABLE:
        return "", [], {"error": "pymupdf_not_available"}
    try:
        doc = fitz.open(path)
        pages = []
        for page in doc[:CONFIG["max_pdf_pages"]]:
            pages.append(normalize_text(page.get_text() or ""))
        meta = {"page_count": doc.page_count, "used_pages": len(pages), "pdf_type": "text_pdf" if any(p.strip() for p in pages) else "scanned_pdf"}
        doc.close()
        return "\f".join(pages), pages, meta
    except Exception as e:
        return "", [], {"error": str(e)}

def render_pdf_pages(path, sid):
    if not PYMUPDF_AVAILABLE:
        return []
    out_dir = DEBUG_DIR / "pdf_pages" / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    out = []
    doc = fitz.open(path)
    for i, page in enumerate(doc[:CONFIG["max_pdf_pages"]], 1):
        pix = page.get_pixmap(matrix=fitz.Matrix(2,2), alpha=False)
        p = out_dir / f"page_{i:03d}.png"
        pix.save(str(p))
        out.append(p)
    doc.close()
    return out

def ocr_candidates_for_image_path(path, sid, page_number=1):
    cands = []
    for variant_name, img in image_preprocess_variants(path, f"{sid}_page{page_number}"):
        for psm in CONFIG["psm_modes"]:
            for lang in CONFIG["tesseract_langs"]:
                cid = f"{sid}_p{page_number}_{variant_name}_psm{psm}_{lang}"
                cands.append(ocr_image_candidate(img, page_number, variant_name, psm, lang, cid))
    if CONFIG["save_variant_texts"]:
        folder = VARIANT_TEXT_DIR / sid
        folder.mkdir(parents=True, exist_ok=True)
        for i, c in enumerate(sorted(cands, key=lambda x: x.score, reverse=True)[:30]):
            (folder / f"{i:02d}_{c.variant_name}_psm{c.psm}_{c.lang}_score{c.score:.3f}.txt").write_text(c.text or "", encoding="utf-8")
    return cands

def get_ocr_candidates_for_file(path, sid):
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        text, pages, meta = extract_pdf_text_layer(path)
        if len(text.strip()) >= CONFIG["min_ocr_text_len"]:
            lines = [{"line_id":i+1,"text":l,"left":None,"top":i,"right":None,"bottom":i,"width":None,"height":None,"word_count":len(l.split()),"avg_conf":0.95}
                     for i,l in enumerate(text.splitlines()) if l.strip()]
            score, status, details = score_ocr_candidate(text, lines, 0.95)
            cand = OCRCandidate(f"{sid}_pdf_text_layer", 1, "pdf_text_layer", None, "embedded", text, 0.95, max(score,0.90), "good_layout_text", {**details, "pdf_meta":meta}, [], lines, None)
            return [cand], meta
        all_cands = []
        for i, img_path in enumerate(render_pdf_pages(path, sid), 1):
            all_cands += ocr_candidates_for_image_path(img_path, sid, i)
        return all_cands, meta
    return ocr_candidates_for_image_path(path, sid, 1), {}

def select_best_candidate(candidates):
    if not candidates:
        return OCRCandidate("empty", 1, "none", None, "none", "", 0, 0, "empty_text", {}, [], [], "no_candidates")
    return sorted(candidates, key=lambda c: (c.score, c.confidence, len(c.text)), reverse=True)[0]


In [8]:
# =========================
# Sections and OCR output rows
# =========================

SECTION_KEYWORDS = {
    "hematology": ["Hematology","CBC","Complete Blood Count","WBC","RBC","Hemoglobin"],
    "biochemistry": ["Biochemistry","Fasting Blood","Glucose","Urea","Creatinine","Cholesterol"],
    "hormone": ["Hormone","Immunoassays","Endocrinology","TSH","Free T4"],
    "urine": ["Urine Analysis","Urine analysts","Macroscopic","Microscopic","Color","Appearance"],
    "culture": ["Culture","Sensitivity","No growth","No bacteria growth"],
}

def detect_sections(lines):
    sections = []
    current = "unknown"
    start_line = 1
    for line in lines:
        low = line["text"].lower()
        detected = None
        for sec, keys in SECTION_KEYWORDS.items():
            if any(k.lower() in low for k in keys):
                detected = sec
                break
        if detected and detected != current:
            if current != "unknown":
                sections.append({"section":current,"start_line":start_line,"end_line":line["line_id"]-1})
            current = detected
            start_line = line["line_id"]
    if lines:
        sections.append({"section":current,"start_line":start_line,"end_line":lines[-1]["line_id"]})
    return sections

def line_to_section(line_id, sections):
    for s in sections:
        if s["start_line"] <= line_id <= s["end_line"]:
            return s["section"]
    return "unknown"

def make_words_lines_sections_tables(doc_index, filename, candidate, source_rel):
    words_rows = []
    for wi, w in enumerate(candidate.words, 1):
        row = dict(w)
        row.update({"index":doc_index,"filename":filename,"candidate_id":candidate.candidate_id,
                    "page_number":candidate.page_number,"variant_name":candidate.variant_name,
                    "psm":candidate.psm,"lang":candidate.lang,"word_index":wi,"source_relative_path":source_rel})
        words_rows.append(row)

    sections = detect_sections(candidate.lines)
    line_rows = []
    for l in candidate.lines:
        row = dict(l)
        row.update({"index":doc_index,"filename":filename,"candidate_id":candidate.candidate_id,
                    "page_number":candidate.page_number,"variant_name":candidate.variant_name,
                    "psm":candidate.psm,"lang":candidate.lang,"section":line_to_section(l["line_id"], sections),
                    "source_relative_path":source_rel})
        line_rows.append(row)

    section_rows = []
    for s in sections:
        row = dict(s)
        row.update({"index":doc_index,"filename":filename,"candidate_id":candidate.candidate_id,"source_relative_path":source_rel})
        section_rows.append(row)
    return words_rows, line_rows, section_rows


In [9]:
# =========================
# Classification and common fields
# =========================

TABLE_TOKENS_RE = re.compile(r"\b(Result|Unit|Reference|Range|WBC|RBC|FBS|TSH|CBC|Biochemistry|Hematology|Platelet|Creatinine|Glucose|Method)\b", re.I)
PATIENT_LABEL_RE = re.compile(r"(نام\s*بیمار|نام\s*مراجعه|Patient\s*Name)", re.I)
DOCTOR_LABEL_RE = re.compile(r"(نام\s*پزشک|پزشک\s*معالج|Doctor|Physician)", re.I)

def is_medical_text(text):
    low = normalize_text(text).lower()
    hits = [k for k in MEDICAL_KEYWORDS if normalize_text(k).lower() in low]
    score = min(1.0, len(hits)/4)
    return score >= 0.15, hits, score

def classify_document(text):
    low = normalize_text(text).lower()
    signals = {
        "lab": ["laboratory","lab","hematology","biochemistry","serology","urine","culture","cbc","wbc","rbc","fbs","glucose","tsh","آزمایشگاه","آزمايشگاه","درمانگاه"],
        "radiology": ["radiology","ultrasound","sonography","mri","ct","x-ray","findings","impression","سونوگرافی","سونوگرافي","رادیولوژی"],
        "pap_smear": ["pap smear","hpv","nilm","ascus","asc-us","lsil","hsil","پاپ اسمیر"],
    }
    best = ("unknown_medical", 0.0, [])
    for dt, sigs in signals.items():
        found = [s for s in sigs if normalize_text(s).lower() in low]
        if len(found) > len(best[2]):
            best = (dt, min(1.0, len(found)/5), found)
    return best

def reject_name_candidate(v):
    if not v: return True
    if len(v) > 90: return True
    if TABLE_TOKENS_RE.search(v): return True
    if len(re.findall(r"\d", v)) > 2: return True
    if re.search(r"پزشک|دکتر|Doctor|Result|Reference|Unit|Method|تاریخ|پذیرش|آزمایشگاه|Laboratory", v, re.I): return True
    return False

def reject_doctor_candidate(v):
    if not v: return True
    if len(v) > 90 or len(v) < 3: return True
    if PATIENT_LABEL_RE.search(v): return True
    if TABLE_TOKENS_RE.search(v): return True
    if re.search(r"آزمایشگاه|Laboratory|پزشکی آزمایشگاه|نام بیمار|بیمار|پذیرش|تاریخ|سن", v, re.I): return True
    if re.search(r"دکتر\s*[0-9]{1,3}", v): return True
    return False

def compact_name_spacing(v):
    v = normalize_text(v)
    v = re.sub(r"(خانم|آقای|آقاي)(?=[آ-ی])", r"\1 ", v)
    return re.sub(r"\s+", " ", v).strip()

def extract_strict_national_id(full):
    m = re.search(r"(?:کد\s*ملی|کد\s*ملي|كد\s*ملی|كد\s*ملي|National\s*ID|National\s*Code)\s*[:：]?\s*([0-9]{10})", full, re.I)
    if m:
        return m.group(1), m.group(0)
    return None, None

def extract_date(full):
    patterns = [
        r"(?:تاریخ\s*پذیرش|تاریخ\s*جواب|تاریخ\s*گزارش|Report\s*Date|Date)\s*[:：]?\s*(?:\d{1,2}:\d{2}:\d{2}\s*[-–]?\s*)?([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})",
        r"([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})\s*(?:0\s*)?(?:تاریخ\s*پذیرش|پذیرش|تاریخ)",
        r"(?:تاریخ\s*پذیرش|پذیرش|تاریخ)\s*[:：]?\s*([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})",
    ]
    for pat in patterns:
        for m in re.finditer(pat, full, re.I):
            cand = normalize_digits(m.group(1))
            if validate_jalali_date(cand):
                return cand, m.group(0)
    return None, None

def extract_tav_header(full):
    m = re.search(r"([^\n]{1,60}?)-\s*(آقای|آقاي|خانم)\s+([^\n]{1,40}?)-\s*دکتر\s*([0-9]{1,3})\s*[:：]\s*سن", full)
    if not m: return {}
    family = clean_value(m.group(1), 50)
    title = m.group(2)
    given = clean_value(m.group(3), 50)
    age = int(m.group(4))
    sex = "female" if title == "خانم" else "male"
    name = clean_value(f"{given} {family}", 90)
    return {"patient_name":(name,m.group(0)), "sex":(sex,m.group(0)), "age":(age,m.group(0))}

def parse_patient_from_line(line):
    line = compact_name_spacing(line)
    if not PATIENT_LABEL_RE.search(line) and "نام بیمار" not in line:
        return None, None, None
    before = re.split(r"نام\s*بیمار|نام\s*مراجعه|Patient\s*Name", line, flags=re.I)[0]
    before = before.replace("+"," ").replace("،"," ").replace(","," ").strip()
    if "خانم" in before or "آقای" in before or "آقاي" in before:
        sex = "female" if "خانم" in before else "male"
        name = before.replace("خانم","").replace("آقای","").replace("آقاي","")
        name = clean_value(name, 90)
        if not reject_name_candidate(name):
            return name, sex, line
    parts = re.split(r"نام\s*بیمار|نام\s*مراجعه|Patient\s*Name", line, flags=re.I)
    if len(parts) > 1:
        after = re.split(r"تاریخ|سن|شماره|پزشک|کد|پذیرش|Date|Age", parts[1])[0]
        sex = "female" if "خانم" in after else ("male" if ("آقای" in after or "آقاي" in after) else None)
        name = after.replace("خانم","").replace("آقای","").replace("آقاي","")
        name = clean_value(name, 90)
        if not reject_name_candidate(name):
            return name, sex, line
    return None, None, None

def extract_compact_patient_name(full):
    pats = [
        r"(?:[0-9]{1,3}\s*سال\s*)?(خانم|آقای|آقاي)\s*([آ-ی]{2,50}(?:\s*[آ-ی]{2,30}){0,3})\s*[:：]?\s*نام",
        r"([آ-ی]{2,40})\s*(خانم|آقای|آقاي)\s*[:：]?\s*(?:ند\s*برد|نام)",
    ]
    for pat in pats:
        m = re.search(pat, full)
        if not m: continue
        if m.group(1) in ["خانم","آقای","آقاي"]:
            sex = "female" if m.group(1) == "خانم" else "male"
            name = clean_value(m.group(2), 90)
        else:
            sex = "female" if m.group(2) == "خانم" else "male"
            name = clean_value(m.group(1), 90)
        if not reject_name_candidate(name):
            return name, sex, m.group(0)
    return None, None, None

def extract_age(full, top):
    for pat in [r"سن\s*[:：]?\s*([0-9]{1,3})(?:\s*سال)?", r"([0-9]{1,3})\s*سال\s*سن", r"سال\s*([0-9]{1,3})\s*سن", r"([0-9]{1,3})\s*سال"]:
        for m in re.finditer(pat, top):
            src = m.group(0)
            if re.search(r"Below age|Above age|Reference|Adults|Children|years?:", src, re.I):
                continue
            val = int(m.group(1))
            if 0 <= val <= 120:
                return val, src
    return None, None

def extract_doctor_from_lines(lines):
    for line in lines[:25]:
        if PATIENT_LABEL_RE.search(line) or not DOCTOR_LABEL_RE.search(line):
            continue
        pieces = re.split(r"نام\s*پزشک|پزشک\s*معالج|پزشک|Doctor|Physician", line, flags=re.I)
        candidates = []
        if len(pieces) > 1: candidates.append(pieces[-1])
        candidates.append(pieces[0])
        for cand in candidates:
            cand = clean_value(cand.replace(":"," ").replace("："," "), 90)
            if cand and not reject_doctor_candidate(cand):
                return cand, line
    return None, None

def extract_center(lines):
    for l in lines[:12]:
        if any(k in l for k in ["آزمایشگاه","آزمايشگاه","Laboratory","درمانگاه","بیمارستان","Nobin"]):
            if TABLE_TOKENS_RE.search(l): continue
            value = clean_value(l, 140)
            if value: return value, l
    return None, None

def validate_center_value(value):
    if not value: return "missing"
    low = value.lower()
    if low.endswith(" and") or len(value) < 6 or re.search(r"\b(and|specialized and)$", low):
        return "review"
    return "valid"

def extract_common_fields(text):
    full = normalize_text(text)
    lines = [l.strip() for l in full.splitlines() if l.strip()]
    top = "\n".join(lines[:25])
    center, center_src = extract_center(lines)
    nid, nid_src = extract_strict_national_id(full)
    tracking = None; tracking_src = None
    for pat in [r"\b([A-Za-z]-\d{5}-\d{3,})\b", r"(?:شماره\s*پذیرش|پذیرش|Admission|Report\s*No|شماره)\s*[:：]?\s*([A-Za-z0-9_-]{3,})", r"([0-9]{4,8})\s*(?:0\s*)?پذیرش"]:
        m = re.search(pat, full, re.I)
        if m:
            tracking, tracking_src = m.group(1), m.group(0); break
    date, date_src = extract_date(full)
    tav = extract_tav_header(full)
    patient_name = sex = age = None
    patient_src = sex_src = age_src = None
    if "patient_name" in tav: patient_name, patient_src = tav["patient_name"]
    if "sex" in tav: sex, sex_src = tav["sex"]
    if "age" in tav: age, age_src = tav["age"]
    if not patient_name:
        for line in lines[:30]:
            cand, sx, src = parse_patient_from_line(line)
            if cand:
                patient_name, patient_src = cand, src
                if sx and not sex: sex, sex_src = sx, src
                break
    if not patient_name:
        cand, sx, src = extract_compact_patient_name(full)
        if cand:
            patient_name, patient_src = cand, src
            if sx and not sex: sex, sex_src = sx, src
    if not sex and patient_src:
        if "خانم" in patient_src or re.search(r"\bFemale\b", patient_src, re.I):
            sex, sex_src = "female", patient_src
        elif "آقای" in patient_src or "آقاي" in patient_src or re.search(r"\bMale\b", patient_src, re.I):
            sex, sex_src = "male", patient_src
    if age is None:
        age, age_src = extract_age(full, top)
    doctor, doctor_src = extract_doctor_from_lines(lines)
    def field(value, conf, src, extra=None):
        d = {"value":value, "confidence":conf if value not in [None,""] else 0.0, "source_text":sanitize_text(src) if src else None}
        if extra: d.update(extra)
        return d
    return {
        "center_name": field(center, 0.7 if validate_center_value(center)=="valid" else 0.55, center_src, {"center_validation_hint":validate_center_value(center)}),
        "national_id": {"value":hash_value(nid), "confidence":0.9 if nid else 0.0, "source_text":sanitize_text(nid_src) if nid_src else None, "strict_label_required":True},
        "tracking_number": field(tracking,0.85,tracking_src),
        "date_of_test_or_report": field(date,0.9,date_src,{"calendar":"jalali" if date else None}),
        "patient_name": field(patient_name,0.88 if patient_name else 0.0,patient_src),
        "sex": field(sex,0.85 if sex else 0.0,sex_src),
        "age": field(age,0.85 if age is not None else 0.0,age_src),
        "doctor_name": field(doctor,0.75 if doctor else 0.0,doctor_src),
    }


In [10]:
# =========================
# Lab parsing and production gate
# =========================

NUM_RE = re.compile(r"(?<!\d)([*<>]?\d+(?:\.\d+)?)(?!\d)")
UNIT_RE = re.compile(r"(10\^?\s*[36]\s*/?\s*(?:uL|µL|μL|pL)|10\*?[36]\s*/?\s*(?:uL|µL|μL|pL)|mg/dL|mg/dl|g/dL|g/dl|U/L|IU/L|mIU/L|µIU/mL|uIU/mL|ng/mL|ng/ml|pg/ml|IU/L|%|fL|fl|pg|Ratio|Sec|mm/hr|mg/d)", re.I)
METHOD_RE = re.compile(r"\b(Photometr|ELISA|ECLIA|CLIA|EIA|HPLC|FLC|KIA\s*&\s*ELFA)\b", re.I)
GUIDELINE_WORDS_RE = re.compile(r"\b(Normal|Borderline|Bordreline|Borrderline|Desirable|High risk|Low risk|Adults?|Males?|Females?|Women|Men|Pregnant|Below age|Above age|Reference|Range|Prognose|Abnormal|Doubtful)\b", re.I)

def alias_regex(alias): return re.escape(alias).replace(r"\ ", r"\s+")
def line_is_known_test(line):
    clean = line.strip()
    for std, alias in ALL_ALIAS_PHRASES:
        if re.fullmatch(alias_regex(alias), clean, re.I):
            return std, alias
    return None, None
def find_test_occurrences(text):
    t = normalize_text(text)
    for std, alias in ALL_ALIAS_PHRASES:
        pat = re.compile(rf"(?i)(?<![A-Za-z]){alias_regex(alias)}(?![A-Za-z])")
        for m in pat.finditer(t):
            yield {"std":std,"alias":alias,"start":m.start(),"end":m.end()}
def next_test_boundary(text, start, min_ahead=5, max_window=240):
    end_limit = min(len(text), start+max_window)
    next_pos = end_limit
    for occ in find_test_occurrences(text[start+min_ahead:end_limit]):
        pos = start + min_ahead + occ["start"]
        if pos > start + min_ahead:
            next_pos = min(next_pos, pos)
    return next_pos
def parse_reference_range(ref):
    if not ref: return None
    r = str(ref).replace(" ","").replace("–","-")
    m = re.match(r"^<\s*(\d+(?:\.\d+)?)$", r)
    if m: return ("upper", float(m.group(1)))
    m = re.match(r"^>\s*(\d+(?:\.\d+)?)$", r)
    if m: return ("lower", float(m.group(1)))
    m = re.match(r"^(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?)$", r)
    if m:
        a,b = float(m.group(1)), float(m.group(2))
        if a <= b: return ("range",a,b)
    return None
def flag_from_reference(value, ref):
    parsed = parse_reference_range(ref)
    if value is None or not parsed: return None
    if parsed[0] == "upper": return "High" if value > parsed[1] else None
    if parsed[0] == "lower": return "Low" if value < parsed[1] else None
    _, lo, hi = parsed
    if value < lo: return "Low"
    if value > hi: return "High"
    return None
def extract_explicit_flag(block, result_end_pos):
    local = block[result_end_pos: result_end_pos+60]
    local = re.sub(r"(High|Low)\s+risk|High\s*[<>]\s*\d+|Low\s*[<>]\s*\d+", "", local, flags=re.I)
    m = re.search(r"(?<![A-Za-z])(High|Low|Critical)(?![A-Za-z])", local, re.I)
    if m: return m.group(1).capitalize()
    m = re.search(r"(?<![A-Za-z.])(H|L)(?![A-Za-z.])", local)
    if m: return "High" if m.group(1)=="H" else "Low"
    if "*" in block[:result_end_pos+20]: return "rechecked"
    return None
def final_flag(value, ref, explicit_flag):
    if parse_reference_range(ref):
        return flag_from_reference(value, ref)
    return explicit_flag
def infer_section(std):
    if std in {"WBC","RBC","HGB","HCT","MCV","MCH","MCHC","RDW-CV","RDW-SD","PLT","PDW","MPV","NEUT%","LYMPH%","MONO%","EO%","BASO%","ESR"}: return "hematology"
    if std in {"PT","PT Control","INR","PTT"}: return "coagulation"
    if std in {"TSH","T3","T4","Free T4","Vitamin D","Ferritin","CRP"}: return "special_biochemistry_or_hormone"
    if std in {"Urine Culture","Color","Appearance","Specific Gravity","pH","Protein","Urine Glucose","Ketone","Bilirubin Urine","Urobilinogen","Nitrite","Blood/Hb","WBC/HPF","RBC/HPF","Bacteria","Mucus","Casts","Crystals"}: return "urine_or_microbiology"
    return "biochemistry"
def try_decimal_correction(std, value, raw):
    corrections=[]; corrected=value
    if std=="MCH" and value>100: corrected=value/10; corrections.append("decimal_shift_divide_by_10_for_MCH")
    elif std=="HbA1c" and value>30: corrected=value/100 if value>100 else value/10; corrections.append("decimal_shift_for_HbA1c")
    elif std=="WBC" and 40<value<200: corrected=value/10; corrections.append("decimal_shift_divide_by_10_for_WBC")
    elif std=="Ferritin" and value>3000 and "." not in str(raw): corrections.append("possible_decimal_missing_for_Ferritin")
    return corrected, corrections
def validate_lab_row(row):
    std = row.get("test_name_standard"); raw=row.get("result_value"); numeric=row.get("result_numeric")
    reasons=[]; validation_status="valid"; save_allowed=True; corrected_value=None
    if numeric is None:
        qv = re.sub(r"\s+"," ",str(raw or "").strip().lower())
        if std in QUALITATIVE_TESTS and any(qv.startswith(x) or x in qv for x in QUALITATIVE_VALID_VALUES):
            return {"validation_status":"valid","save_allowed":True,"reason_codes":[],"corrected_numeric":None,"corrected_value":None}
        return {"validation_status":"review","save_allowed":False,"reason_codes":["non_numeric_or_unrecognized_textual_result"],"corrected_numeric":None,"corrected_value":None}
    corrected_numeric, corrections = try_decimal_correction(std, float(numeric), str(raw))
    if corrections:
        reasons += corrections
        if std in ROW_SANITY_LIMITS:
            lo,hi=ROW_SANITY_LIMITS[std]
            if lo <= corrected_numeric <= hi:
                validation_status="corrected_review"; save_allowed=False; corrected_value=str(corrected_numeric)
    check = corrected_numeric if corrected_value is not None else float(numeric)
    if std in ROW_SANITY_LIMITS:
        lo,hi=ROW_SANITY_LIMITS[std]
        if not (lo <= check <= hi):
            validation_status="invalid"; save_allowed=False; reasons.append(f"outside_physiologic_range_{lo}_{hi}")
    if row.get("confidence",0) < 0.60:
        if validation_status=="valid": validation_status="review"
        save_allowed=False; reasons.append("low_row_confidence")
    if row.get("unit") is None and std not in QUALITATIVE_TESTS and std not in {"pH"}:
        if validation_status=="valid": validation_status="review"
        save_allowed=False; reasons.append("missing_unit")
    return {"validation_status":validation_status,"save_allowed":save_allowed,"reason_codes":sorted(set(reasons)),"corrected_numeric":corrected_numeric if corrected_value is not None else None,"corrected_value":corrected_value}
def parse_test_block(std, alias, block, confidence_base=0.70):
    b = block.strip()
    b_after = re.sub(rf"(?i)^\s*{alias_regex(alias)}\s*", "", b).strip()
    num_match = None
    for m in NUM_RE.finditer(b_after[:120]):
        before = b_after[max(0,m.start()-30):m.start()]
        if GUIDELINE_WORDS_RE.search(before) and m.start()>25: continue
        if re.search(r"\d{4}[/-]\d", b_after[max(0,m.start()-10):m.end()+12]): continue
        num_match=m; break
    if not num_match: return None
    result_raw = num_match.group(1).lstrip("*")
    try: numeric=float(re.sub(r"^[<>]+","",result_raw))
    except Exception: numeric=None
    after = b_after[num_match.end():]
    unit_m = UNIT_RE.search(after[:70])
    unit = unit_m.group(0) if unit_m else None
    ref=None
    ref_m = re.search(r"([<>]\s*\d+(?:\.\d+)?|\d+(?:\.\d+)?\s*[-–]\s*\d+(?:\.\d+)?)", after[:100])
    if ref_m: ref=re.sub(r"\s+","",ref_m.group(1))
    method_m = METHOD_RE.search(after[:140])
    explicit = extract_explicit_flag(b_after, num_match.end())
    row = {"section":infer_section(std),"test_name_standard":std,"test_name_raw":alias,"result_value":result_raw,"result_numeric":numeric,
           "unit":unit,"reference_range":ref,"flag":final_flag(numeric,ref,explicit),"method":method_m.group(1) if method_m else None,
           "confidence":confidence_base,"source_text":sanitize_text(b[:300])}
    row.update(validate_lab_row(row)); return row
def extract_clean_multiline_blocks(text):
    lines=[l.strip() for l in normalize_text(text).splitlines() if l.strip()]
    rows=[]; i=0
    while i < len(lines):
        std,alias=line_is_known_test(lines[i])
        if not std: i+=1; continue
        j=i+1; block_lines=[lines[i]]
        while j < len(lines):
            nstd,_=line_is_known_test(lines[j])
            if nstd: break
            if not re.fullmatch(r"(Test|Result|Unit|Normal Range|Reference Range|Flag|Method|Biochemistry|Hematology)", lines[j], re.I):
                block_lines.append(lines[j])
            if len(block_lines)>=8: break
            j+=1
        row=parse_test_block(std,alias,"\n".join(block_lines),0.82)
        if row: rows.append(row)
        i=max(j,i+1)
    return rows
def extract_messy_window_rows(text):
    flat=re.sub(r"\s+"," ",normalize_text(text)); rows=[]
    for occ in sorted(find_test_occurrences(flat), key=lambda x:x["start"]):
        start=occ["start"]; end=next_test_boundary(flat,start,max_window=210)
        row=parse_test_block(occ["std"],occ["alias"],flat[start:end],0.58)
        if row: rows.append(row)
    return rows
def extract_urine_culture(text):
    flat=re.sub(r"\s+"," ",normalize_text(text)); rows=[]
    m=re.search(r"(No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*(?:hrs?|hours)\.?)", flat, re.I)
    if m:
        row={"section":"microbiology","test_name_standard":"Urine Culture","test_name_raw":"Culture","result_value":m.group(1),"result_numeric":None,"unit":None,"reference_range":None,"flag":None,"method":None,"confidence":0.88,"source_text":sanitize_text(flat[max(0,m.start()-80):m.end()+80])}
        row.update(validate_lab_row(row)); rows.append(row)
    q_patterns={"Color":r"Color\s+(Yellow|Amber|Red|Pale|Straw|Yeon|Yetlow)","Appearance":r"Appearance\s+(Clear|Semi[- ]?Clear|Turbid|Cloudy|Rac)","Protein":r"Protein\s+(Negative|Positive|Trace|[0-9+]+)","Urine Glucose":r"Glucose\s+(Negative|Positive|Trace|[0-9+]+)","Ketone":r"Ketone\s+(Negative|Positive|Trace|[0-9+]+)","Nitrite":r"Nitrite\s+(Negative|Positive)","Bacteria":r"Bacteria\s+(Negative|Not seen|Look at culture|Few|Rare|Many)","Mucus":r"Mucus\s+(Negative|Not seen|Few|Rare|Many)","Casts":r"Casts?\s+(Negative|Not seen|Few|Rare|Many)","Crystals":r"Crystals?\s+(Negative|Not seen|Few|Rare|Many)"}
    for std,pat in q_patterns.items():
        m=re.search(pat,flat,re.I)
        if m:
            row={"section":"urine_analysis","test_name_standard":std,"test_name_raw":std,"result_value":m.group(1),"result_numeric":None,"unit":None,"reference_range":None,"flag":None,"method":None,"confidence":0.66,"source_text":sanitize_text(flat[max(0,m.start()-80):m.end()+80])}
            row.update(validate_lab_row(row)); rows.append(row)
    return rows
def dedup_lab(rows):
    best={}
    for r in rows:
        key=(r.get("test_name_standard"),str(r.get("result_value")).lower(),r.get("unit"))
        if key not in best or r.get("confidence",0)>best[key].get("confidence",0): best[key]=r
    return list(best.values())
def extract_lab_results(text):
    return dedup_lab(extract_clean_multiline_blocks(text)+extract_messy_window_rows(text)+extract_urine_culture(text))


In [11]:
# =========================
# Backend safety policy
# =========================

def validate_common_fields(common, doc_type, backend_patient_context_available=False):
    field_status={}; reasons=[]
    for k,v in common.items():
        value=v.get("value"); conf=v.get("confidence",0)
        if value in [None,""]: field_status[k]="missing"
        elif k=="center_name" and v.get("center_validation_hint")=="review": field_status[k]="review"
        elif conf<0.7: field_status[k]="review"
        else: field_status[k]="valid"
    has=lambda k: common.get(k,{}).get("value") not in [None,""]
    if CONFIG["require_report_date_for_backend_save"] and not has("date_of_test_or_report"): reasons.append("missing_report_date")
    patient_ok = has("patient_name") or has("national_id") or backend_patient_context_available
    if CONFIG["require_patient_or_backend_context_for_backend_save"] and not patient_ok: reasons.append("missing_patient_identity_or_backend_context")
    return {"field_validation_status":field_status,"critical_field_reason_codes":reasons,"critical_fields_valid_for_backend":len(reasons)==0}
def valid_common_payload(common, field_validation_status):
    payload={}
    for k,v in common.items():
        if field_validation_status.get(k)=="valid" and v.get("value") is not None:
            payload[k]={"value":v.get("value"),"confidence":v.get("confidence"),"source_text":v.get("source_text")}
            if "calendar" in v: payload[k]["calendar"]=v.get("calendar")
            if k=="national_id": payload[k]["is_hash"]=True
    return payload
def validate_rows_for_backend(labs):
    counts=Counter(); safe=[]; unsafe=[]
    for r in labs:
        st=r.get("validation_status","review"); counts[st]+=1
        if r.get("save_allowed"): safe.append(r)
        else: unsafe.append({"test_name_standard":r.get("test_name_standard"),"result_value":r.get("result_value"),"validation_status":st,"reason_codes":r.get("reason_codes",[]),"source_text":r.get("source_text")})
    return {"row_validation_status":dict(counts),"safe_row_count":len(safe),"unsafe_row_count":len(unsafe),"unsafe_rows_sample":unsafe[:50],"safe_rows":safe}
def decide_extraction_status(path, quality, chosen, medical_ok, doc_type, common, labs):
    if quality.get("status") in ["poor_quality","unreadable"] and Path(path).suffix.lower()!=".pdf": return "poor_quality_rejected", ["input_quality_too_poor"]
    if not chosen.text.strip(): return "ocr_failed", ["ocr_failed"]
    if chosen.status in ["poor_ocr_text","gibberish_or_bad_layout_text"] and not chosen.score_details.get("culture_signal"): return "ocr_unreliable_needs_reupload", [f"ocr_layout_status={chosen.status}"]
    if not medical_ok: return "unrelated_or_not_medical", ["no_medical_signals"]
    if doc_type=="lab":
        if not labs: return "partial_extraction", ["no_lab_rows_extracted"]
        return "extracted_good", []
    return "needs_manual_review", ["unknown_or_weak_document_type"]
def build_backend_save_policy(extraction_status, common, common_validation, row_validation, doc_type):
    reasons=[]; route="save_to_database"; allowed=True
    if extraction_status in ["poor_quality_rejected","ocr_failed","ocr_unreliable_needs_reupload"]:
        allowed=False; route="ask_user_reupload"; reasons.append(extraction_status)
    if extraction_status=="unrelated_or_not_medical":
        allowed=False; route="reject_unrelated"; reasons.append("unrelated_or_not_medical")
    if extraction_status in ["partial_extraction","needs_manual_review"]:
        allowed=False; route="manual_review"; reasons.append(extraction_status)
    if not common_validation["critical_fields_valid_for_backend"]:
        allowed=False
        if route=="save_to_database": route="manual_review"
        reasons += common_validation["critical_field_reason_codes"]
    if row_validation["safe_row_count"]==0 and doc_type=="lab":
        allowed=False
        if route=="save_to_database": route="manual_review"
        reasons.append("no_safe_lab_rows")
    if row_validation["unsafe_row_count"]>0: reasons.append("some_rows_require_review")
    payload={"common_fields":valid_common_payload(common, common_validation["field_validation_status"]),"lab_results":row_validation["safe_rows"]}
    if not allowed: payload={"common_fields":{},"lab_results":[]}
    return {"backend_save_allowed":allowed,"review_required":route=="manual_review","reupload_required":route=="ask_user_reupload","route":route,"reason_codes":sorted(set(reasons)),"safe_payload":payload}
def common_to_parts(common, field_validation_status):
    return [{"part_type":"common_field","section":"header","field_name":k,"value":v.get("value"),"confidence":v.get("confidence",0),"source_text":v.get("source_text"),"field_validation_status":field_validation_status.get(k,"unknown")} for k,v in common.items() if v.get("value") is not None]
def lab_to_parts(labs):
    rows=[]
    for r in labs:
        rows.append({"part_type":"lab_result","section":r.get("section"),"field_name":r.get("test_name_standard"),"value":r.get("result_value"),"result_numeric":r.get("result_numeric"),"unit":r.get("unit"),"reference_range":r.get("reference_range"),"flag":r.get("flag"),"method":r.get("method"),"confidence":r.get("confidence"),"source_text":r.get("source_text"),"test_name_raw":r.get("test_name_raw"),"row_validation_status":r.get("validation_status"),"row_save_allowed":r.get("save_allowed"),"row_reason_codes":"|".join(r.get("reason_codes",[])),"corrected_numeric":r.get("corrected_numeric"),"corrected_value":r.get("corrected_value")})
    return rows


In [12]:

# =========================
# v10: Column-aware extraction, statuses, and safer scoring
# =========================

# ---- Priority 4: alias collision control ----
# Remove dangerous broad aliases that caused false rows.
LAB_ALIASES["Blood/Hb"] = ["Blood/Hb"]                         # Do not use generic "Blood" or "Hemoglobin"
LAB_ALIASES["Urine Glucose"] = ["Urine Glucose", "Glucose"]     # Glucose only allowed in urine context
LAB_ALIASES["Color"] = ["Color"]
LAB_ALIASES["Appearance"] = ["Appearance"]
LAB_ALIASES["MCH"] = ["MCH", "M.C.H"]                           # Must not match MCHC
LAB_ALIASES["MCHC"] = ["MCHC", "M.C.H.C"]

ALL_ALIAS_PHRASES = sorted(
    [(std, alias) for std, aliases in LAB_ALIASES.items() for alias in aliases],
    key=lambda x: len(x[1]),
    reverse=True,
)
ALL_ALIAS_LOWER = [(std, alias, alias.lower()) for std, alias in ALL_ALIAS_PHRASES]

URINE_CONTEXT_TESTS = {
    "Urine Glucose", "Color", "Appearance", "Specific Gravity", "pH", "Protein",
    "Ketone", "Bilirubin Urine", "Urobilinogen", "Nitrite", "Blood/Hb",
    "WBC/HPF", "RBC/HPF", "Bacteria", "Mucus", "Casts", "Crystals",
}

REQUIRED_COMMON_FIELDS_FOR_BACKEND = ["date_of_test_or_report"]
PATIENT_IDENTITY_FIELDS = ["patient_name", "national_id"]

FIELD_STATUS_ALLOWED = {
    "valid",
    "review",
    "missing",
    "missing_required",
    "missing_optional",
    "invalid",
    "unsafe_ocr_context",
    "not_applicable",
}

COLUMN_STATUS_ALLOWED = {
    "valid",
    "review",
    "missing_required",
    "missing_optional",
    "invalid",
    "not_applicable",
    "unsafe_ocr_context",
}

def strict_alias_regex(std, alias):
    escaped = re.escape(alias).replace(r"\ ", r"\s+")
    if std == "MCH":
        return rf"(?<![A-Za-z0-9]){escaped}(?!\s*C)(?![A-Za-z0-9])"
    if std == "MCHC":
        return rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])"
    if len(alias) <= 2:
        return rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])"
    return rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])"

def alias_allowed_in_context(std, alias, line_text, section):
    section = section or "unknown"
    low = normalize_text(line_text).lower()

    # Urine-only fields must be in urine/culture section or have an explicit urine context.
    if std in URINE_CONTEXT_TESTS:
        urine_signal = section in {"urine", "culture", "urine_or_microbiology"} or "urine" in low or "macroscopic" in low or "microscopic" in low
        if std in {"Color", "Appearance", "Urine Glucose", "Blood/Hb"} and not urine_signal:
            return False
        if std == "Urine Glucose" and alias.lower() == "glucose" and not urine_signal:
            return False
        if std == "Blood/Hb" and "blood/hb" not in low and not urine_signal:
            return False

    # Prevent blood sugar from becoming Blood/Hb.
    if std == "Blood/Hb" and "blood sugar" in low:
        return False

    # Prevent generic Color from non-urine/report title areas.
    if std == "Color" and section not in {"urine", "urine_or_microbiology"} and "urine" not in low:
        return False

    return True

def find_strict_test_occurrences_in_text(text, section="unknown"):
    t = normalize_text(text)
    out = []
    occupied = []
    for std, alias in ALL_ALIAS_PHRASES:
        pat = re.compile(strict_alias_regex(std, alias), re.I)
        for m in pat.finditer(t):
            if not alias_allowed_in_context(std, alias, t, section):
                continue
            # avoid overlaps from shorter aliases after longer aliases
            if any(not (m.end() <= a or m.start() >= b) for a, b in occupied):
                continue
            occupied.append((m.start(), m.end()))
            out.append({"std": std, "alias": alias, "start": m.start(), "end": m.end()})
    return sorted(out, key=lambda x: x["start"])

def alias_hits(text):
    hits = []
    for occ in find_strict_test_occurrences_in_text(text):
        hits.append((occ["std"], occ["alias"]))
    return hits

# ---- Priority 2: graph/history-area removal ----
def is_graph_or_history_line(text):
    t = normalize_text(text)
    low = t.lower()

    if re.search(r"\b(graph|chart|trend|history|previous|delta|diff)\b", low):
        return True

    # Many dates or date-like fragments often belong to charts/history blocks.
    date_like = re.findall(r"\b(?:13|14)\d{2}[/-]\d{1,2}[/-]\d{1,2}\b|\b\d{2}[/-]\d{2}[/-]\d{2,4}\b", t)
    if len(date_like) >= 2:
        return True

    # Many separated numbers with no strong test alias/unit -> likely graph axis/history area.
    nums = re.findall(r"(?<!\d)\d+(?:\.\d+)?(?!\d)", t)
    has_alias = bool(find_strict_test_occurrences_in_text(t))
    has_unit = bool(UNIT_RE.search(t))
    if len(nums) >= 6 and not has_alias and not has_unit:
        return True

    # Repeated tiny fragments around old results.
    if len(nums) >= 5 and re.search(r"\b(20|21|22|23|24|25|26|27|28|29|30)\b", t) and not has_unit:
        return True

    return False

# ---- Priority 3: stricter common-field extraction and validation ----
def extract_age(full, top):
    full = normalize_text(full)
    top = normalize_text(top)

    # strongest: label and age in one phrase
    patterns = [
        r"سن\s*[:：]?\s*([0-9]{1,3})\s*سال",
        r"([0-9]{1,3})\s*سال\s*[:：]?\s*سن",
        r"سال\s*([0-9]{1,3})\s*[:：]?\s*سن",
        r"Age\s*[:：]?\s*([0-9]{1,3})",
        r"سن\s*[:：]?\s*([0-9]{1,3})(?!\d)",
    ]
    search_area = top[:1800]
    for pat in patterns:
        for m in re.finditer(pat, search_area, re.I):
            src = m.group(0)
            if re.search(r"Below age|Above age|Reference|Adults|Children|years?:|range", src, re.I):
                continue
            val = int(m.group(1))
            if 0 <= val <= 120:
                return val, src

    # TAV reversed style: "دکتر34 : سن"
    m = re.search(r"دکتر\s*([0-9]{1,3})\s*[:：]\s*سن", search_area)
    if m:
        val = int(m.group(1))
        if 0 <= val <= 120:
            return val, m.group(0)

    return None, None

def clean_doctor_name_v10(value):
    if not value:
        return None
    v = normalize_text(value)
    v = re.sub(r"\b0\b", " ", v)
    v = re.sub(r"آقای\s*دکتر|جناب\s*آقای\s*دکتر|دکتر\s*آقای", "دکتر", v)
    v = re.sub(r"Doctor|Physician|Dr\.", "دکتر", v, flags=re.I)
    v = re.sub(r"\s+", " ", v).strip(" :：،,؛")
    if "دکتر" in v:
        # keep short phrase around doctor name
        m = re.search(r"(?:دکتر\s*)?([آ-ی]{2,20})\s+([آ-ی]{2,20})(?:\s+دکتر)?", v)
        if m:
            return f"دکتر {m.group(1)} {m.group(2)}"
    return clean_value(v, 80)

def extract_doctor_from_lines(lines):
    for line in lines[:35]:
        if PATIENT_LABEL_RE.search(line):
            continue
        if not DOCTOR_LABEL_RE.search(line) and "دکتر" not in line:
            continue
        pieces = re.split(r"نام\s*پزشک|پزشک\s*معالج|پزشک|Doctor|Physician", line, flags=re.I)
        candidates = []
        if len(pieces) > 1:
            candidates.append(pieces[-1])
        candidates.append(pieces[0])
        candidates.append(line)
        for cand in candidates:
            cand = clean_doctor_name_v10(cand)
            if cand and not reject_doctor_candidate(cand):
                return cand, line
    return None, None

def validate_common_fields_v10(common, doc_type, ocr_layout_status, backend_patient_context_available=False):
    field_status = {}
    reason_codes = []

    unsafe_ocr = ocr_layout_status in {"poor_ocr_text", "gibberish_or_bad_layout_text", "empty_text"}

    for k, v in common.items():
        value = v.get("value")
        conf = float(v.get("confidence", 0) or 0)

        if value in [None, ""]:
            if k in REQUIRED_COMMON_FIELDS_FOR_BACKEND or k in PATIENT_IDENTITY_FIELDS:
                field_status[k] = "missing_required" if k in REQUIRED_COMMON_FIELDS_FOR_BACKEND else "missing"
            else:
                field_status[k] = "missing_optional"
            continue

        if unsafe_ocr:
            field_status[k] = "unsafe_ocr_context"
            continue

        if k == "date_of_test_or_report":
            field_status[k] = "valid" if validate_jalali_date(str(value)) else "invalid"
            continue

        if k == "age":
            try:
                age = int(value)
                src = normalize_text(v.get("source_text") or "")
                if 0 <= age <= 120 and ("سن" in src or "Age" in src or "سال" in src):
                    field_status[k] = "valid"
                else:
                    field_status[k] = "review"
            except Exception:
                field_status[k] = "invalid"
            continue

        if k == "patient_name":
            val = normalize_text(str(value))
            if reject_name_candidate(val):
                field_status[k] = "invalid"
            elif len(val.split()) < 2 and len(val) < 8:
                field_status[k] = "review"
            else:
                field_status[k] = "valid"
            continue

        if k == "doctor_name":
            val = normalize_text(str(value))
            if reject_doctor_candidate(val):
                field_status[k] = "invalid"
            else:
                field_status[k] = "valid" if conf >= 0.70 else "review"
            continue

        if k == "center_name" and v.get("center_validation_hint") == "review":
            field_status[k] = "review"
        elif conf < 0.70:
            field_status[k] = "review"
        else:
            field_status[k] = "valid"

    # Critical validation
    date_status = field_status.get("date_of_test_or_report")
    if CONFIG["require_report_date_for_backend_save"] and date_status != "valid":
        reason_codes.append("missing_or_invalid_report_date")

    patient_identity_ok = (
        field_status.get("patient_name") == "valid"
        or field_status.get("national_id") == "valid"
        or backend_patient_context_available
    )
    if CONFIG["require_patient_or_backend_context_for_backend_save"] and not patient_identity_ok:
        reason_codes.append("missing_or_invalid_patient_identity_or_backend_context")

    if unsafe_ocr:
        reason_codes.append("unsafe_ocr_context_for_common_fields")

    return {
        "field_validation_status": field_status,
        "critical_field_reason_codes": sorted(set(reason_codes)),
        "critical_fields_valid_for_backend": len(reason_codes) == 0,
    }

# ---- Priority 6 + Product-team requirement: per-column status ----
def parse_reference_range_v10(ref):
    return parse_reference_range(ref)

def qualitative_result_valid(std, value):
    qv = re.sub(r"\s+", " ", str(value or "").strip().lower())
    return std in QUALITATIVE_TESTS and any(qv.startswith(x) or x in qv for x in QUALITATIVE_VALID_VALUES)

def column_statuses_for_row(row):
    std = row.get("test_name_standard")
    numeric = row.get("result_numeric")
    value = row.get("result_value")
    unit = row.get("unit")
    ref = row.get("reference_range")
    flag = row.get("flag")
    method = row.get("method")

    statuses = {
        "test_name_status": "valid" if std else "missing_required",
        "result_status": "missing_required",
        "unit_status": "not_applicable",
        "reference_range_status": "missing_optional",
        "flag_status": "not_applicable",
        "method_status": "missing_optional",
    }

    if numeric is not None:
        statuses["result_status"] = "valid"
        if std in ROW_SANITY_LIMITS:
            lo, hi = ROW_SANITY_LIMITS[std]
            if not (lo <= float(numeric) <= hi):
                statuses["result_status"] = "invalid"
        statuses["unit_status"] = "valid" if unit else "missing_required"
    else:
        if qualitative_result_valid(std, value):
            statuses["result_status"] = "valid"
            statuses["unit_status"] = "not_applicable"
        else:
            statuses["result_status"] = "review" if value else "missing_required"

    if ref:
        statuses["reference_range_status"] = "valid" if parse_reference_range_v10(ref) else "review"
    if flag:
        statuses["flag_status"] = "valid" if str(flag).lower() in {"high", "low", "critical", "rechecked"} else "review"
    if method:
        statuses["method_status"] = "valid"

    return statuses

def validate_lab_row_v10(row, layout_confidence=0.60, section_context="unknown"):
    std = row.get("test_name_standard")
    raw = row.get("result_value")
    numeric = row.get("result_numeric")
    reasons = []
    validation_status = "valid"
    save_allowed = True
    corrected_value = None

    corrected_numeric = None
    if numeric is not None:
        corrected_numeric, corrections = try_decimal_correction(std, float(numeric), str(raw))
        if corrections:
            reasons.extend(corrections)
            if std in ROW_SANITY_LIMITS:
                lo, hi = ROW_SANITY_LIMITS[std]
                if lo <= corrected_numeric <= hi:
                    validation_status = "corrected_review"
                    save_allowed = False
                    corrected_value = str(corrected_numeric)

    column_statuses = column_statuses_for_row(row)

    # Invalid required columns block save.
    for k, st in column_statuses.items():
        if st in {"invalid", "missing_required"}:
            validation_status = "invalid" if st == "invalid" else "review"
            save_allowed = False
            reasons.append(f"{k}:{st}")

    # Confidence now depends on layout evidence, not a fixed 0.58.
    row_conf = float(row.get("confidence", 0) or 0)
    if row_conf < 0.65:
        if validation_status == "valid":
            validation_status = "review"
        save_allowed = False
        reasons.append("low_layout_row_confidence")

    # Urine-context fields outside urine section are review, not safe.
    if std in URINE_CONTEXT_TESTS and section_context not in {"urine", "culture", "urine_or_microbiology"}:
        if std not in {"WBC/HPF", "RBC/HPF"}:
            if validation_status == "valid":
                validation_status = "review"
            save_allowed = False
            reasons.append("urine_field_outside_urine_section")

    row["column_statuses"] = column_statuses
    row["test_name_status"] = column_statuses["test_name_status"]
    row["result_status"] = column_statuses["result_status"]
    row["unit_status"] = column_statuses["unit_status"]
    row["reference_range_status"] = column_statuses["reference_range_status"]
    row["flag_status"] = column_statuses["flag_status"]
    row["method_status"] = column_statuses["method_status"]

    return {
        "validation_status": validation_status,
        "save_allowed": save_allowed,
        "reason_codes": sorted(set(reasons)),
        "corrected_numeric": corrected_numeric if corrected_value is not None else None,
        "corrected_value": corrected_value,
        "column_statuses": column_statuses,
    }

def layout_row_confidence(line_text, alias, value, unit, ref, word_conf=0.60, same_visual_line=True):
    score = 0.40
    if alias:
        score += 0.15
    if value not in [None, ""]:
        score += 0.15
    if unit:
        score += 0.10
    if ref:
        score += 0.08
    if same_visual_line:
        score += 0.07
    score += min(max(float(word_conf or 0), 0), 1) * 0.20
    if is_graph_or_history_line(line_text):
        score -= 0.25
    return round(max(0.0, min(0.98, score)), 3)

# ---- Priority 1: column-aware row extraction from visual lines/word boxes ----
def parse_value_unit_ref_from_segment(segment):
    segment = normalize_text(segment)
    num_m = NUM_RE.search(segment[:120])
    if not num_m:
        # qualitative immediate value
        q = re.search(r"\b(Negative|Positive|Trace|Clear|Yellow|Amber|Red|Pale|Straw|Normal|Not seen|Few|Rare|Many|Acid|Alkaline|Look at culture)\b", segment, re.I)
        if q:
            return q.group(1), None, None, None, q.end()
        return None, None, None, None, 0

    raw = num_m.group(1).lstrip("*")
    try:
        numeric = float(re.sub(r"^[<>]+", "", raw))
    except Exception:
        numeric = None

    after = segment[num_m.end():]
    unit_m = UNIT_RE.search(after[:80])
    unit = unit_m.group(0) if unit_m else None

    ref = None
    ref_m = re.search(r"([<>]\s*\d+(?:\.\d+)?|\d+(?:\.\d+)?\s*[-–]\s*\d+(?:\.\d+)?)", after[:120])
    if ref_m:
        ref = re.sub(r"\s+", "", ref_m.group(1))

    method_m = METHOD_RE.search(after[:160])
    method = method_m.group(1) if method_m else None
    return raw, numeric, unit, ref, num_m.end()

def parse_column_line_rows(line, section):
    text = normalize_text(line.get("text", ""))
    if not text or is_graph_or_history_line(text):
        return []

    occurrences = find_strict_test_occurrences_in_text(text, section)
    if not occurrences:
        return []

    rows = []
    for idx, occ in enumerate(occurrences):
        start = occ["end"]
        end = occurrences[idx + 1]["start"] if idx + 1 < len(occurrences) else len(text)
        segment = text[start:end].strip(" :：|-")
        value, numeric, unit, ref, pos = parse_value_unit_ref_from_segment(segment)
        if value is None:
            continue

        method_m = METHOD_RE.search(segment[:160])
        explicit_flag = extract_explicit_flag(segment, pos)
        flag = final_flag(numeric, ref, explicit_flag)

        conf = layout_row_confidence(
            text, occ["alias"], value, unit, ref,
            word_conf=line.get("avg_conf", 0.60),
            same_visual_line=True,
        )

        row = {
            "section": infer_section(occ["std"]) if section == "unknown" else section,
            "test_name_standard": occ["std"],
            "test_name_raw": occ["alias"],
            "result_value": value,
            "result_numeric": numeric,
            "unit": unit,
            "reference_range": ref,
            "flag": flag,
            "method": method_m.group(1) if method_m else None,
            "confidence": conf,
            "source_text": sanitize_text(text[:300]),
            "extraction_mode": "column_aware_visual_line",
            "line_id": line.get("line_id"),
            "layout_evidence": {
                "same_visual_line": True,
                "line_avg_conf": line.get("avg_conf"),
                "section": section,
                "left": line.get("left"),
                "top": line.get("top"),
                "width": line.get("width"),
                "height": line.get("height"),
            },
        }
        row.update(validate_lab_row_v10(row, layout_confidence=conf, section_context=section))
        rows.append(row)

    return rows

def extract_column_aware_rows_from_lines(lines):
    if not lines:
        return []
    sections = detect_sections(lines)
    rows = []
    for line in lines:
        section = line_to_section(line.get("line_id"), sections)
        # Fix urine subsection contamination from WBC/RBC keywords.
        if "Urine" in normalize_text(line.get("text", "")) or section == "urine":
            section = "urine"
        rows.extend(parse_column_line_rows(line, section))
    return rows

def parse_test_block_v10(std, alias, block, confidence_base=0.72, section_context="unknown"):
    b = block.strip()
    if is_graph_or_history_line(b):
        return None
    b_after = re.sub(rf"(?i)^\s*{alias_regex(alias)}\s*", "", b).strip()
    value, numeric, unit, ref, pos = parse_value_unit_ref_from_segment(b_after)
    if value is None:
        return None
    method_m = METHOD_RE.search(b_after[:180])
    explicit = extract_explicit_flag(b_after, pos)
    flag = final_flag(numeric, ref, explicit)
    row = {
        "section": infer_section(std) if section_context == "unknown" else section_context,
        "test_name_standard": std,
        "test_name_raw": alias,
        "result_value": value,
        "result_numeric": numeric,
        "unit": unit,
        "reference_range": ref,
        "flag": flag,
        "method": method_m.group(1) if method_m else None,
        "confidence": confidence_base,
        "source_text": sanitize_text(b[:300]),
        "extraction_mode": "strict_multiline_or_window",
    }
    row.update(validate_lab_row_v10(row, layout_confidence=confidence_base, section_context=row["section"]))
    return row

def extract_clean_multiline_blocks_v10(text):
    t = normalize_text(text)
    lines = [l.strip() for l in t.splitlines() if l.strip()]
    rows = []
    i = 0
    while i < len(lines):
        line = lines[i]
        if is_graph_or_history_line(line):
            i += 1
            continue

        occs = find_strict_test_occurrences_in_text(line)
        # only treat exact/single test-name lines as a multiline block
        if not occs or len(line) > max(len(occs[0]["alias"]) + 8, 30):
            i += 1
            continue

        occ = occs[0]
        j = i + 1
        block_lines = [line]
        while j < len(lines):
            next_occs = find_strict_test_occurrences_in_text(lines[j])
            if next_occs and len(lines[j]) <= max(len(next_occs[0]["alias"]) + 8, 30):
                break
            if not is_graph_or_history_line(lines[j]):
                block_lines.append(lines[j])
            if len(block_lines) >= 8:
                break
            j += 1

        row = parse_test_block_v10(occ["std"], occ["alias"], "\n".join(block_lines), 0.82)
        if row:
            rows.append(row)
        i = max(j, i + 1)

    return rows

def extract_messy_window_rows_v10(text):
    flat = re.sub(r"\s+", " ", normalize_text(text))
    rows = []
    for occ in find_strict_test_occurrences_in_text(flat):
        start = occ["start"]
        end = min(len(flat), start + 210)
        for nxt in find_strict_test_occurrences_in_text(flat[start + 5:end]):
            end = start + 5 + nxt["start"]
            break
        block = flat[start:end]
        row = parse_test_block_v10(occ["std"], occ["alias"], block, 0.60)
        if row:
            rows.append(row)
    return rows

def extract_urine_culture_v10(text):
    rows = []
    flat = re.sub(r"\s+", " ", normalize_text(text))

    m = re.search(r"(No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*(?:hrs?|hours)\.?)", flat, re.I)
    if m:
        row = {
            "section": "culture",
            "test_name_standard": "Urine Culture",
            "test_name_raw": "Culture",
            "result_value": m.group(1),
            "result_numeric": None,
            "unit": None,
            "reference_range": None,
            "flag": None,
            "method": None,
            "confidence": 0.92,
            "source_text": sanitize_text(flat[max(0, m.start() - 80):m.end() + 80]),
            "extraction_mode": "culture_phrase",
        }
        row.update(validate_lab_row_v10(row, layout_confidence=0.92, section_context="culture"))
        rows.append(row)

    qualitative_patterns = {
        "Color": r"Color\s+(Yellow|Amber|Red|Pale|Straw|Yeon|Yetlow)",
        "Appearance": r"Appearance\s+(Clear|Semi[- ]?Clear|Turbid|Cloudy|Rac)",
        "Protein": r"Protein\s+(Negative|Positive|Trace|[0-9+]+)",
        "Urine Glucose": r"(?:Urine\s+)?Glucose\s+(Negative|Positive|Trace|[0-9+]+)",
        "Ketone": r"Ketone\s+(Negative|Positive|Trace|[0-9+]+)",
        "Bilirubin Urine": r"Bilirubin\s+(Negative|Positive|Trace|[0-9+]+)",
        "Urobilinogen": r"Urobilinogen\s+(Normal|Negative|Positive|[0-9.]+)",
        "Nitrite": r"Nitrite\s+(Negative|Positive)",
        "Blood/Hb": r"(?:Blood/Hb)\s+(Negative|Positive|Trace|[0-9+]+)",
        "Bacteria": r"Bacteria\s+(Negative|Not seen|Look at culture|Few|Rare|Many)",
        "Mucus": r"Mucus\s+(Negative|Not seen|Few|Rare|Many)",
        "Casts": r"Casts?\s+(Negative|Not seen|Few|Rare|Many)",
        "Crystals": r"Crystals?\s+(Negative|Not seen|Few|Rare|Many)",
    }
    urine_context = bool(re.search(r"Urine|Macroscopic|Microscopic|Bacteria|Nitrite", flat, re.I))
    if urine_context:
        for std, pat in qualitative_patterns.items():
            m = re.search(pat, flat, re.I)
            if not m:
                continue
            row = {
                "section": "urine",
                "test_name_standard": std,
                "test_name_raw": std,
                "result_value": m.group(1),
                "result_numeric": None,
                "unit": None,
                "reference_range": None,
                "flag": None,
                "method": None,
                "confidence": 0.74,
                "source_text": sanitize_text(flat[max(0, m.start() - 80):m.end() + 80]),
                "extraction_mode": "urine_qualitative_pattern",
            }
            row.update(validate_lab_row_v10(row, layout_confidence=0.74, section_context="urine"))
            rows.append(row)

    return rows

def dedup_lab_v10(rows):
    best = {}
    for r in rows:
        key = (
            r.get("test_name_standard"),
            str(r.get("result_value")).lower(),
            r.get("unit"),
            r.get("section"),
        )
        if key not in best or r.get("confidence", 0) > best[key].get("confidence", 0):
            best[key] = r

    # Drop low-confidence duplicate alias collisions when a higher-confidence standard exists.
    out = list(best.values())
    filtered = []
    for r in out:
        std = r.get("test_name_standard")
        src = normalize_text(r.get("source_text") or "")
        if std == "MCH" and re.search(r"M\.?C\.?H\.?C", src, re.I):
            continue
        if std == "Blood/Hb" and re.search(r"blood sugar|Fasting blood", src, re.I):
            continue
        if std == "Urine Glucose" and r.get("section") not in {"urine", "urine_or_microbiology"}:
            continue
        filtered.append(r)
    return filtered

def extract_lab_results_v10(text, lines=None):
    rows = []
    if lines:
        rows.extend(extract_column_aware_rows_from_lines(lines))
    rows.extend(extract_clean_multiline_blocks_v10(text))
    rows.extend(extract_messy_window_rows_v10(text))
    rows.extend(extract_urine_culture_v10(text))
    return dedup_lab_v10(rows)

# ---- Priority 5: candidate scoring uses trial extraction and safety signal ----
def vertical_line_penalty(lines):
    if not lines:
        return 0.0
    bad = 0
    for l in lines:
        w = l.get("width") or 0
        h = l.get("height") or 0
        if h > max(80, w * 2.5):
            bad += 1
    return min(0.25, bad / max(1, len(lines)) * 0.7)

def suspicious_common_penalty(common):
    penalty = 0.0
    age = common.get("age", {}).get("value")
    age_src = normalize_text(common.get("age", {}).get("source_text") or "")
    if age is not None:
        try:
            a = int(age)
            if not (0 <= a <= 120) or ("سن" not in age_src and "Age" not in age_src and "سال" not in age_src):
                penalty += 0.10
        except Exception:
            penalty += 0.10

    patient = common.get("patient_name", {}).get("value")
    if patient and reject_name_candidate(str(patient)):
        penalty += 0.10

    return penalty

def select_best_candidate_v10(candidates, quality):
    if not candidates:
        return OCRCandidate("empty", 1, "none", None, "none", "", 0, 0, "empty_text", {}, [], [], "no_candidates")

    evaluated = []
    for c in candidates:
        text = normalize_text(c.text)
        common = extract_common_fields(text)
        labs = extract_lab_results_v10(text, c.lines)
        row_val = validate_rows_for_backend_v10(labs)
        doc_type, _, _ = classify_document(text)

        safe_rows = row_val["safe_row_count"]
        unsafe_rows = row_val["unsafe_row_count"]
        total_rows = safe_rows + unsafe_rows

        common_bonus = 0.0
        if common.get("date_of_test_or_report", {}).get("value"):
            common_bonus += 0.04
        if common.get("patient_name", {}).get("value") or common.get("national_id", {}).get("value"):
            common_bonus += 0.04

        row_bonus = min(0.16, safe_rows * 0.015)
        invalid_ratio_penalty = 0.0
        if total_rows:
            invalid_ratio_penalty = min(0.18, unsafe_rows / total_rows * 0.12)

        vertical_pen = vertical_line_penalty(c.lines)
        suspicious_pen = suspicious_common_penalty(common)

        # Good OCR that produces no safe rows should not be overtrusted for images.
        no_safe_row_penalty = 0.08 if doc_type == "lab" and safe_rows == 0 and len(text) > 300 else 0.0

        v10_score = (
            float(c.score)
            + common_bonus
            + row_bonus
            - invalid_ratio_penalty
            - vertical_pen
            - suspicious_pen
            - no_safe_row_penalty
        )
        v10_score = max(0.0, min(1.0, v10_score))

        c.score_details = dict(c.score_details)
        c.score_details.update({
            "v10_selection_score": round(v10_score, 3),
            "v10_safe_row_count": safe_rows,
            "v10_unsafe_row_count": unsafe_rows,
            "v10_common_bonus": round(common_bonus, 3),
            "v10_row_bonus": round(row_bonus, 3),
            "v10_invalid_ratio_penalty": round(invalid_ratio_penalty, 3),
            "v10_vertical_line_penalty": round(vertical_pen, 3),
            "v10_suspicious_common_penalty": round(suspicious_pen, 3),
            "v10_no_safe_row_penalty": round(no_safe_row_penalty, 3),
        })
        evaluated.append((v10_score, c.confidence, len(text), c))

    return sorted(evaluated, key=lambda x: (x[0], x[1], x[2]), reverse=True)[0][3]

# Override output row conversion to include column statuses.
def lab_to_parts_v10(labs):
    rows = []
    for r in labs:
        column_statuses = r.get("column_statuses") or column_statuses_for_row(r)
        rows.append({
            "part_type": "lab_result",
            "section": r.get("section"),
            "field_name": r.get("test_name_standard"),
            "value": r.get("result_value"),
            "result_numeric": r.get("result_numeric"),
            "unit": r.get("unit"),
            "reference_range": r.get("reference_range"),
            "flag": r.get("flag"),
            "method": r.get("method"),
            "confidence": r.get("confidence"),
            "source_text": r.get("source_text"),
            "test_name_raw": r.get("test_name_raw"),
            "extraction_mode": r.get("extraction_mode"),
            "row_validation_status": r.get("validation_status"),
            "row_save_allowed": r.get("save_allowed"),
            "row_reason_codes": "|".join(r.get("reason_codes", [])),
            "corrected_numeric": r.get("corrected_numeric"),
            "corrected_value": r.get("corrected_value"),
            "column_statuses_json": json.dumps(column_statuses, ensure_ascii=False),
            "test_name_status": column_statuses.get("test_name_status"),
            "result_status": column_statuses.get("result_status"),
            "unit_status": column_statuses.get("unit_status"),
            "reference_range_status": column_statuses.get("reference_range_status"),
            "flag_status": column_statuses.get("flag_status"),
            "method_status": column_statuses.get("method_status"),
        })
    return rows

def validate_rows_for_backend_v10(labs):
    counts = Counter()
    column_counts = Counter()
    unsafe = []
    safe = []

    for r in labs:
        status = r.get("validation_status", "review")
        counts[status] += 1
        for k, v in (r.get("column_statuses") or {}).items():
            column_counts[f"{k}:{v}"] += 1

        if r.get("save_allowed"):
            safe.append(r)
        else:
            unsafe.append({
                "test_name_standard": r.get("test_name_standard"),
                "result_value": r.get("result_value"),
                "validation_status": status,
                "column_statuses": r.get("column_statuses"),
                "reason_codes": r.get("reason_codes", []),
                "source_text": r.get("source_text"),
            })

    return {
        "row_validation_status": dict(counts),
        "column_status_counts": dict(column_counts),
        "safe_row_count": len(safe),
        "unsafe_row_count": len(unsafe),
        "unsafe_rows_sample": unsafe[:80],
        "safe_rows": safe,
    }

def build_backend_save_policy_v10(extraction_status, common, common_validation, row_validation, doc_type):
    return build_backend_save_policy(extraction_status, common, common_validation, row_validation, doc_type)

print("v10 overrides loaded: column-aware extraction, field/column statuses, safer candidate scoring.")


v10 overrides loaded: column-aware extraction, field/column statuses, safer candidate scoring.


In [13]:

# =========================
# v11: Production safety overrides
# =========================

# ---- 5) Expanded physiological ranges ----
ROW_SANITY_LIMITS.update({
    "BUN": (2.0, 150.0),
    "Urea": (5.0, 300.0),
    "Total Cholesterol": (50.0, 600.0),
    "Triglycerides": (20.0, 2000.0),
    "HDL": (5.0, 150.0),
    "LDL": (10.0, 400.0),
    "AST": (1.0, 1000.0),
    "ALT": (1.0, 1000.0),
    "ESR": (0.0, 150.0),
    "PT": (5.0, 80.0),
    "PT Control": (5.0, 30.0),
    "INR": (0.5, 10.0),
    "PTT": (10.0, 250.0),
    "CRP": (0.0, 500.0),
})

EXPECTED_UNITS = {
    "WBC": {"10^3/uL", "10^3/µL", "10^3/μL", "10*3/uL", "10*3/µL", "1000/cumm", "k/ul", "10³/µL", "10°/pL"},
    "RBC": {"10^6/uL", "10^6/µL", "10^6/μL", "10*6/uL", "10*6/µL", "mil/cumm", "m/ul", "10⁶/µL"},
    "HGB": {"g/dL", "g/dl"},
    "HCT": {"%"},
    "MCV": {"fL", "fl"},
    "MCH": {"pg"},
    "MCHC": {"g/dL", "g/dl"},
    "PLT": {"10^3/uL", "10^3/µL", "10^3/μL", "10*3/uL", "10*3/µL", "1000/cumm", "k/ul", "10³/µL", "10°/pL"},
    "RDW-CV": {"%"},
    "NEUT%": {"%"},
    "LYMPH%": {"%"},
    "MONO%": {"%"},
    "EO%": {"%"},
    "BASO%": {"%"},
    "HbA1c": {"%"},
    "FBS": {"mg/dL", "mg/dl"},
    "BUN": {"mg/dL", "mg/dl"},
    "Urea": {"mg/dL", "mg/dl"},
    "Creatinine": {"mg/dL", "mg/dl"},
    "Uric Acid": {"mg/dL", "mg/dl"},
    "Total Cholesterol": {"mg/dL", "mg/dl"},
    "Triglycerides": {"mg/dL", "mg/dl"},
    "HDL": {"mg/dL", "mg/dl"},
    "LDL": {"mg/dL", "mg/dl"},
    "AST": {"U/L", "IU/L"},
    "ALT": {"U/L", "IU/L"},
    "ALP": {"U/L", "IU/L"},
    "Ferritin": {"ng/mL", "ng/ml"},
    "Vitamin D": {"ng/mL", "ng/ml"},
    "TSH": {"µIU/mL", "uIU/mL", "mIU/L"},
    "CRP": {"mg/L", "mg/dL", "mg/dl"},
    "PT": {"Sec", "sec", "s"},
    "PT Control": {"Sec", "sec", "s"},
    "INR": {"Ratio", ""},
    "PTT": {"Sec", "sec", "s"},
    "ESR": {"mm/hr", "mm/h"},
}

def normalize_unit_for_check(unit):
    if unit is None:
        return None
    u = str(unit).strip()
    u = u.replace("μ", "µ")
    u = re.sub(r"\s+", "", u)
    u = u.replace("10^3", "10^3").replace("10*3", "10^3").replace("10³", "10^3")
    u = u.replace("10^6", "10^6").replace("10*6", "10^6").replace("10⁶", "10^6")
    return u.lower()

def expected_unit_status(std, unit):
    if std not in EXPECTED_UNITS:
        return "not_applicable", None
    if unit in [None, ""]:
        # qualitative tests handled elsewhere; most numeric lab rows need unit
        return "missing_required", "expected_unit_missing"

    allowed = EXPECTED_UNITS[std]
    allowed_norm = {normalize_unit_for_check(x) for x in allowed if x != ""}
    u = normalize_unit_for_check(unit)

    # tolerate OCR pL for µL in CBC if otherwise same 10^n pattern
    if std in {"WBC", "PLT"} and u and ("10^3" in u or "10°" in u):
        return "valid", None
    if std == "RBC" and u and "10^6" in u:
        return "valid", None

    if u in allowed_norm:
        return "valid", None

    return "invalid", f"unexpected_unit:{unit}:expected:{sorted(allowed)}"

# ---- 6) stricter tracking number ----
def is_valid_tracking_number(value, source=None):
    if value in [None, ""]:
        return False, "missing"
    v = normalize_text(str(value)).strip()
    s = normalize_text(str(source or ""))
    if re.fullmatch(r"(Result|Hormone|Biochemistry|Hematology|Urine|Culture|Method|Reference)", v, re.I):
        return False, "label_text_not_tracking"
    if re.fullmatch(r"13\d{2}|14\d{2}", v):
        return False, "year_not_tracking"
    if validate_jalali_date(v):
        return False, "date_not_tracking"
    if len(v) < 5:
        return False, "too_short"
    if not re.search(r"(پذیرش|Admission|Report|شماره|No)", s, re.I):
        # unlabeled pure alphanumeric ids are review, not valid
        return False, "not_near_tracking_label"
    if re.search(r"[A-Za-z]-\d{5}-\d{3,}", v):
        return True, "valid"
    if re.fullmatch(r"[A-Za-z0-9_-]{5,20}", v):
        return True, "valid"
    return False, "invalid_format"

# ---- 1) doctor only with explicit label ----
EXPLICIT_DOCTOR_LABEL_RE = re.compile(r"(نام\s*پزشک|پزشک\s*معالج|درخواست\s*کننده|Doctor|Physician|Lab\.?\s*Director|دکتر\s*مسئول)", re.I)

def extract_doctor_from_lines(lines):
    # No more accepting a random "دکتر" from patient/age line.
    for line in lines[:35]:
        if PATIENT_LABEL_RE.search(line):
            continue
        if "سن" in normalize_text(line):
            continue
        if not EXPLICIT_DOCTOR_LABEL_RE.search(line):
            continue

        pieces = re.split(
            r"نام\s*پزشک|پزشک\s*معالج|درخواست\s*کننده|Doctor|Physician|Lab\.?\s*Director|دکتر\s*مسئول",
            line,
            flags=re.I,
        )
        candidates = []
        if len(pieces) > 1:
            candidates.append(pieces[-1])
        # reversed value before label only if it does not look like patient/age
        candidates.append(pieces[0])

        for cand in candidates:
            cand = clean_doctor_name_v10(cand)
            if cand and not reject_doctor_candidate(cand):
                return cand, line, True
    return None, None, False

def extract_common_fields_v11(text):
    full = normalize_text(text)
    lines = [l.strip() for l in full.splitlines() if l.strip()]
    top = "\n".join(lines[:25])

    center, center_src = extract_center(lines)
    nid, nid_src = extract_strict_national_id(full)

    tracking = None
    tracking_src = None
    for pat in [
        r"\b([A-Za-z]-\d{5}-\d{3,})\b",
        r"(?:شماره\s*پذیرش|پذیرش|Admission|Report\s*No|شماره)\s*[:：]?\s*([A-Za-z0-9_-]{5,20})",
    ]:
        m = re.search(pat, full, re.I)
        if m:
            ok, _reason = is_valid_tracking_number(m.group(1), m.group(0))
            if ok:
                tracking, tracking_src = m.group(1), m.group(0)
                break

    date, date_src = extract_date(full)

    tav = extract_tav_header(full)
    patient_name = sex = age = None
    patient_src = sex_src = age_src = None

    if "patient_name" in tav:
        patient_name, patient_src = tav["patient_name"]
    if "sex" in tav:
        sex, sex_src = tav["sex"]
    if "age" in tav:
        age, age_src = tav["age"]

    if not patient_name:
        for line in lines[:30]:
            cand, sx, src = parse_patient_from_line(line)
            if cand:
                patient_name, patient_src = cand, src
                if sx and not sex:
                    sex, sex_src = sx, src
                break

    if not patient_name:
        cand, sx, src = extract_compact_patient_name(full)
        if cand:
            patient_name, patient_src = cand, src
            if sx and not sex:
                sex, sex_src = sx, src

    if not sex and patient_src:
        if "خانم" in patient_src or re.search(r"\bFemale\b", patient_src, re.I):
            sex, sex_src = "female", patient_src
        elif "آقای" in patient_src or "آقاي" in patient_src or re.search(r"\bMale\b", patient_src, re.I):
            sex, sex_src = "male", patient_src

    if age is None:
        age, age_src = extract_age(full, top)

    doctor, doctor_src, explicit_doctor_label = extract_doctor_from_lines(lines)

    def field(value, conf, src, extra=None):
        d = {
            "value": value,
            "confidence": conf if value not in [None, ""] else 0.0,
            "source_text": sanitize_text(src) if src else None,
        }
        if extra:
            d.update(extra)
        return d

    tracking_valid, tracking_reason = is_valid_tracking_number(tracking, tracking_src)

    return {
        "center_name": field(center, 0.7 if validate_center_value(center) == "valid" else 0.55, center_src, {"center_validation_hint": validate_center_value(center)}),
        "national_id": {"value": hash_value(nid), "confidence": 0.9 if nid else 0.0, "source_text": sanitize_text(nid_src) if nid_src else None, "strict_label_required": True},
        "tracking_number": field(tracking if tracking_valid else None, 0.85 if tracking_valid else 0.0, tracking_src, {"tracking_validation_reason": tracking_reason}),
        "date_of_test_or_report": field(date, 0.9, date_src, {"calendar": "jalali" if date else None}),
        "patient_name": field(patient_name, 0.88 if patient_name else 0.0, patient_src),
        "sex": field(sex, 0.85 if sex else 0.0, sex_src),
        "age": field(age, 0.85 if age is not None else 0.0, age_src),
        "doctor_name": field(doctor, 0.75 if doctor else 0.0, doctor_src, {"explicit_doctor_label": explicit_doctor_label}),
    }

# Override common extractor used in batch and scoring
extract_common_fields = extract_common_fields_v11

# ---- 7) Culture parser: no Bacteria=48 / Culture=48 false rows ----
def parse_column_line_rows_v11(line, section):
    text = normalize_text(line.get("text", ""))
    if not text or is_graph_or_history_line(text):
        return []

    # If the line is a culture sentence, only extract the culture phrase row.
    if culture_signal(text):
        return []

    occurrences = find_strict_test_occurrences_in_text(text, section)
    if not occurrences:
        return []

    rows = []
    for idx, occ in enumerate(occurrences):
        std = occ["std"]

        # Culture and Bacteria need textual/qualitative extraction, not number extraction from "48 hrs".
        if std in {"Urine Culture", "Bacteria"} and re.search(r"growth\s+after\s+\d+", text, re.I):
            continue

        start = occ["end"]
        end = occurrences[idx + 1]["start"] if idx + 1 < len(occurrences) else len(text)
        segment = text[start:end].strip(" :：|-")
        value, numeric, unit, ref, pos = parse_value_unit_ref_from_segment(segment)

        if value is None:
            continue

        method_m = METHOD_RE.search(segment[:160])
        explicit_flag = extract_explicit_flag(segment, pos)
        flag = final_flag(numeric, ref, explicit_flag)

        conf = layout_row_confidence(
            text,
            occ["alias"],
            value,
            unit,
            ref,
            word_conf=line.get("avg_conf", 0.60),
            same_visual_line=True,
        )

        row = {
            "section": infer_section(std) if section == "unknown" else section,
            "test_name_standard": std,
            "test_name_raw": occ["alias"],
            "result_value": value,
            "result_numeric": numeric,
            "unit": unit,
            "reference_range": ref,
            "flag": flag,
            "method": method_m.group(1) if method_m else None,
            "confidence": conf,
            "source_text": sanitize_text(text[:300]),
            "extraction_mode": "column_aware_visual_line",
            "line_id": line.get("line_id"),
            "layout_evidence": {
                "same_visual_line": True,
                "line_avg_conf": line.get("avg_conf"),
                "section": section,
                "left": line.get("left"),
                "top": line.get("top"),
                "width": line.get("width"),
                "height": line.get("height"),
            },
        }
        row.update(validate_lab_row_v11(row, layout_confidence=conf, section_context=section))
        rows.append(row)

    return rows

def extract_column_aware_rows_from_lines_v11(lines):
    if not lines:
        return []
    sections = detect_sections(lines)
    rows = []
    for line in lines:
        section = line_to_section(line.get("line_id"), sections)
        txt = normalize_text(line.get("text", ""))
        if "Urine" in txt or section == "urine":
            section = "urine"
        if culture_signal(txt):
            continue
        rows.extend(parse_column_line_rows_v11(line, section))
    return rows

def extract_urine_culture_v11(text):
    rows = []
    flat = re.sub(r"\s+", " ", normalize_text(text))

    m = re.search(r"(No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*(?:hrs?|hours)\.?)", flat, re.I)
    if m:
        row = {
            "section": "culture",
            "test_name_standard": "Urine Culture",
            "test_name_raw": "Culture",
            "result_value": m.group(1),
            "result_numeric": None,
            "unit": None,
            "reference_range": None,
            "flag": None,
            "method": None,
            "confidence": 0.95,
            "source_text": sanitize_text(flat[max(0, m.start() - 80):m.end() + 80]),
            "extraction_mode": "culture_phrase_only",
        }
        row.update(validate_lab_row_v11(row, layout_confidence=0.95, section_context="culture"))
        rows.append(row)

    # Only run urine qualitative patterns when there is explicit urine context.
    urine_context = bool(re.search(r"Urine|Macroscopic|Microscopic|WBC/HPF|RBC/HPF|Nitrite|Urobilinogen", flat, re.I))
    if not urine_context:
        return rows

    qualitative_patterns = {
        "Color": r"Color\s+(Yellow|Amber|Red|Pale|Straw|Yeon|Yetlow)",
        "Appearance": r"Appearance\s+(Clear|Semi[- ]?Clear|Turbid|Cloudy|Rac)",
        "Protein": r"Protein\s+(Negative|Positive|Trace|[0-9+]+)",
        "Urine Glucose": r"(?:Urine\s+)?Glucose\s+(Negative|Positive|Trace|[0-9+]+)",
        "Ketone": r"Ketone\s+(Negative|Positive|Trace|[0-9+]+)",
        "Bilirubin Urine": r"Bilirubin\s+(Negative|Positive|Trace|[0-9+]+)",
        "Urobilinogen": r"Urobilinogen\s+(Normal|Negative|Positive|[0-9.]+)",
        "Nitrite": r"Nitrite\s+(Negative|Positive)",
        "Blood/Hb": r"(?:Blood/Hb)\s+(Negative|Positive|Trace|[0-9+]+)",
        "Bacteria": r"Bacteria\s+(Negative|Not seen|Look at culture|Few|Rare|Many)",
        "Mucus": r"Mucus\s+(Negative|Not seen|Few|Rare|Many)",
        "Casts": r"Casts?\s+(Negative|Not seen|Few|Rare|Many)",
        "Crystals": r"Crystals?\s+(Negative|Not seen|Few|Rare|Many)",
    }

    for std, pat in qualitative_patterns.items():
        m = re.search(pat, flat, re.I)
        if not m:
            continue
        # Prevent Bacteria=48 style false rows.
        if std == "Bacteria" and re.search(r"growth\s+after\s+\d+", flat[max(0, m.start()-40):m.end()+40], re.I):
            continue
        row = {
            "section": "urine",
            "test_name_standard": std,
            "test_name_raw": std,
            "result_value": m.group(1),
            "result_numeric": None,
            "unit": None,
            "reference_range": None,
            "flag": None,
            "method": None,
            "confidence": 0.76,
            "source_text": sanitize_text(flat[max(0, m.start() - 80):m.end() + 80]),
            "extraction_mode": "urine_qualitative_pattern",
        }
        row.update(validate_lab_row_v11(row, layout_confidence=0.76, section_context="urine"))
        rows.append(row)

    return rows

# ---- 3/4/5: stricter row/column validation with expected units ----
def column_statuses_for_row_v11(row, unsafe_ocr_context=False):
    std = row.get("test_name_standard")
    numeric = row.get("result_numeric")
    value = row.get("result_value")
    unit = row.get("unit")
    ref = row.get("reference_range")
    flag = row.get("flag")
    method = row.get("method")

    if unsafe_ocr_context:
        return {
            "test_name_status": "unsafe_ocr_context" if std else "missing_required",
            "result_status": "unsafe_ocr_context" if value is not None else "missing_required",
            "unit_status": "unsafe_ocr_context" if unit else "missing_optional",
            "reference_range_status": "unsafe_ocr_context" if ref else "missing_optional",
            "flag_status": "unsafe_ocr_context" if flag else "not_applicable",
            "method_status": "unsafe_ocr_context" if method else "missing_optional",
        }

    statuses = {
        "test_name_status": "valid" if std else "missing_required",
        "result_status": "missing_required",
        "unit_status": "not_applicable",
        "reference_range_status": "missing_optional",
        "flag_status": "not_applicable",
        "method_status": "missing_optional",
    }

    if numeric is not None:
        statuses["result_status"] = "valid"
        if std in ROW_SANITY_LIMITS:
            lo, hi = ROW_SANITY_LIMITS[std]
            if not (lo <= float(numeric) <= hi):
                statuses["result_status"] = "invalid"

        unit_st, _unit_reason = expected_unit_status(std, unit)
        statuses["unit_status"] = unit_st

    else:
        if qualitative_result_valid(std, value):
            statuses["result_status"] = "valid"
            statuses["unit_status"] = "not_applicable"
        else:
            statuses["result_status"] = "review" if value else "missing_required"

    if ref:
        statuses["reference_range_status"] = "valid" if parse_reference_range_v10(ref) else "review"
    if flag:
        statuses["flag_status"] = "valid" if str(flag).lower() in {"high", "low", "critical", "rechecked"} else "review"
    if method:
        statuses["method_status"] = "valid"

    return statuses

def validate_lab_row_v11(row, layout_confidence=0.60, section_context="unknown", unsafe_ocr_context=False):
    std = row.get("test_name_standard")
    raw = row.get("result_value")
    numeric = row.get("result_numeric")
    reasons = []
    validation_status = "valid"
    save_allowed = True
    corrected_value = None
    corrected_numeric = None

    if unsafe_ocr_context:
        column_statuses = column_statuses_for_row_v11(row, unsafe_ocr_context=True)
        row.update({
            "column_statuses": column_statuses,
            "test_name_status": column_statuses["test_name_status"],
            "result_status": column_statuses["result_status"],
            "unit_status": column_statuses["unit_status"],
            "reference_range_status": column_statuses["reference_range_status"],
            "flag_status": column_statuses["flag_status"],
            "method_status": column_statuses["method_status"],
        })
        return {
            "validation_status": "unsafe_ocr_context",
            "save_allowed": False,
            "reason_codes": ["unsafe_ocr_context"],
            "corrected_numeric": None,
            "corrected_value": None,
            "column_statuses": column_statuses,
        }

    if numeric is not None:
        corrected_numeric, corrections = try_decimal_correction(std, float(numeric), str(raw))
        if corrections:
            reasons.extend(corrections)
            if std in ROW_SANITY_LIMITS:
                lo, hi = ROW_SANITY_LIMITS[std]
                if lo <= corrected_numeric <= hi:
                    validation_status = "corrected_review"
                    save_allowed = False
                    corrected_value = str(corrected_numeric)

    column_statuses = column_statuses_for_row_v11(row, unsafe_ocr_context=False)

    for k, st in column_statuses.items():
        if st == "invalid":
            validation_status = "invalid"
            save_allowed = False
            reasons.append(f"{k}:invalid")
        elif st == "missing_required":
            if validation_status == "valid":
                validation_status = "review"
            save_allowed = False
            reasons.append(f"{k}:missing_required")
        elif st == "review":
            if validation_status == "valid":
                validation_status = "review"
            save_allowed = False
            reasons.append(f"{k}:review")

    # Expected unit validation reason.
    unit_st, unit_reason = expected_unit_status(std, row.get("unit"))
    if unit_reason:
        reasons.append(unit_reason)

    row_conf = float(row.get("confidence", 0) or 0)
    if row_conf < 0.68:
        if validation_status == "valid":
            validation_status = "review"
        save_allowed = False
        reasons.append("low_layout_row_confidence")

    if std in URINE_CONTEXT_TESTS and section_context not in {"urine", "culture", "urine_or_microbiology"}:
        if std not in {"WBC/HPF", "RBC/HPF"}:
            if validation_status == "valid":
                validation_status = "review"
            save_allowed = False
            reasons.append("urine_field_outside_urine_section")

    row.update({
        "column_statuses": column_statuses,
        "test_name_status": column_statuses["test_name_status"],
        "result_status": column_statuses["result_status"],
        "unit_status": column_statuses["unit_status"],
        "reference_range_status": column_statuses["reference_range_status"],
        "flag_status": column_statuses["flag_status"],
        "method_status": column_statuses["method_status"],
    })

    return {
        "validation_status": validation_status,
        "save_allowed": save_allowed,
        "reason_codes": sorted(set(reasons)),
        "corrected_numeric": corrected_numeric if corrected_value is not None else None,
        "corrected_value": corrected_value,
        "column_statuses": column_statuses,
    }

# Override parse/extraction methods to use v11 validators.
def parse_test_block_v11(std, alias, block, confidence_base=0.72, section_context="unknown", unsafe_ocr_context=False):
    b = block.strip()
    if is_graph_or_history_line(b) or culture_signal(b):
        return None
    b_after = re.sub(rf"(?i)^\s*{alias_regex(alias)}\s*", "", b).strip()
    value, numeric, unit, ref, pos = parse_value_unit_ref_from_segment(b_after)
    if value is None:
        return None
    method_m = METHOD_RE.search(b_after[:180])
    explicit = extract_explicit_flag(b_after, pos)
    flag = final_flag(numeric, ref, explicit)
    row = {
        "section": infer_section(std) if section_context == "unknown" else section_context,
        "test_name_standard": std,
        "test_name_raw": alias,
        "result_value": value,
        "result_numeric": numeric,
        "unit": unit,
        "reference_range": ref,
        "flag": flag,
        "method": method_m.group(1) if method_m else None,
        "confidence": confidence_base,
        "source_text": sanitize_text(b[:300]),
        "extraction_mode": "strict_multiline_or_window",
    }
    row.update(validate_lab_row_v11(row, layout_confidence=confidence_base, section_context=row["section"], unsafe_ocr_context=unsafe_ocr_context))
    return row

def extract_clean_multiline_blocks_v11(text, unsafe_ocr_context=False):
    t = normalize_text(text)
    lines = [l.strip() for l in t.splitlines() if l.strip()]
    rows = []
    i = 0
    while i < len(lines):
        line = lines[i]
        if is_graph_or_history_line(line) or culture_signal(line):
            i += 1
            continue
        occs = find_strict_test_occurrences_in_text(line)
        if not occs or len(line) > max(len(occs[0]["alias"]) + 8, 30):
            i += 1
            continue
        occ = occs[0]
        j = i + 1
        block_lines = [line]
        while j < len(lines):
            next_occs = find_strict_test_occurrences_in_text(lines[j])
            if next_occs and len(lines[j]) <= max(len(next_occs[0]["alias"]) + 8, 30):
                break
            if not is_graph_or_history_line(lines[j]) and not culture_signal(lines[j]):
                block_lines.append(lines[j])
            if len(block_lines) >= 8:
                break
            j += 1
        row = parse_test_block_v11(occ["std"], occ["alias"], "\n".join(block_lines), 0.82, unsafe_ocr_context=unsafe_ocr_context)
        if row:
            rows.append(row)
        i = max(j, i + 1)
    return rows

def extract_messy_window_rows_v11(text, unsafe_ocr_context=False):
    flat = re.sub(r"\s+", " ", normalize_text(text))
    rows = []
    for occ in find_strict_test_occurrences_in_text(flat):
        start = occ["start"]
        end = min(len(flat), start + 210)
        snippet = flat[start:end]
        if culture_signal(snippet):
            continue
        for nxt in find_strict_test_occurrences_in_text(flat[start + 5:end]):
            end = start + 5 + nxt["start"]
            break
        block = flat[start:end]
        row = parse_test_block_v11(occ["std"], occ["alias"], block, 0.62, unsafe_ocr_context=unsafe_ocr_context)
        if row:
            rows.append(row)
    return rows

def dedup_lab_v11(rows):
    best = {}
    for r in rows:
        key = (
            r.get("test_name_standard"),
            str(r.get("result_value")).lower(),
            r.get("unit"),
            r.get("section"),
        )
        if key not in best or r.get("confidence", 0) > best[key].get("confidence", 0):
            best[key] = r

    filtered = []
    for r in best.values():
        std = r.get("test_name_standard")
        src = normalize_text(r.get("source_text") or "")
        if std == "MCH" and re.search(r"M\.?C\.?H\.?C", src, re.I):
            continue
        if std == "Blood/Hb" and re.search(r"blood sugar|Fasting blood", src, re.I):
            continue
        if std == "Urine Glucose" and r.get("section") not in {"urine", "urine_or_microbiology"}:
            continue
        if std in {"Bacteria", "Urine Culture"} and re.search(r"growth\s+after\s+\d+", src, re.I) and r.get("extraction_mode") != "culture_phrase_only":
            continue
        filtered.append(r)
    return filtered

def extract_lab_results_v11(text, lines=None, unsafe_ocr_context=False):
    rows = []
    if lines and not unsafe_ocr_context:
        rows.extend(extract_column_aware_rows_from_lines_v11(lines))
    rows.extend(extract_clean_multiline_blocks_v11(text, unsafe_ocr_context=unsafe_ocr_context))
    rows.extend(extract_messy_window_rows_v11(text, unsafe_ocr_context=unsafe_ocr_context))
    rows.extend(extract_urine_culture_v11(text))
    if unsafe_ocr_context:
        # force every row to unsafe context even if a pattern matched
        forced = []
        for r in rows:
            r.update(validate_lab_row_v11(r, unsafe_ocr_context=True))
            forced.append(r)
        rows = forced
    return dedup_lab_v11(rows)

# ---- 3) Force unsafe OCR context after route decision ----
def force_rows_unsafe_ocr_context(rows):
    forced = []
    for r in rows:
        r = dict(r)
        r.update(validate_lab_row_v11(r, unsafe_ocr_context=True))
        forced.append(r)
    return forced

def force_common_unsafe_context(common, common_validation):
    cv = dict(common_validation)
    statuses = dict(cv.get("field_validation_status", {}))
    for k, v in common.items():
        if v.get("value") not in [None, ""]:
            statuses[k] = "unsafe_ocr_context"
    cv["field_validation_status"] = statuses
    reasons = set(cv.get("critical_field_reason_codes", []))
    reasons.add("unsafe_ocr_context_for_common_fields")
    cv["critical_field_reason_codes"] = sorted(reasons)
    cv["critical_fields_valid_for_backend"] = False
    return cv

# ---- 2/8) backend row save allowed ----
def add_backend_row_flags(rows, backend_save_allowed):
    out = []
    for r in rows:
        r = dict(r)
        r["backend_row_save_allowed"] = bool(backend_save_allowed and r.get("save_allowed"))
        if not r["backend_row_save_allowed"] and backend_save_allowed is False:
            # row may be internally valid for review, but cannot be saved because document is blocked
            reasons = set(r.get("reason_codes", []))
            reasons.add("document_not_backend_save_allowed")
            r["backend_reason_codes"] = sorted(reasons)
        else:
            r["backend_reason_codes"] = r.get("reason_codes", [])
        out.append(r)
    return out

# ---- 1) safe payload: doctor_name only if explicit label ----
def valid_common_payload_v11(common, field_validation_status):
    payload = {}
    for k, v in common.items():
        if field_validation_status.get(k) != "valid" or v.get("value") is None:
            continue
        if k == "doctor_name" and not v.get("explicit_doctor_label"):
            continue
        payload[k] = {
            "value": v.get("value"),
            "confidence": v.get("confidence"),
            "source_text": v.get("source_text"),
        }
        if "calendar" in v:
            payload[k]["calendar"] = v.get("calendar")
        if k == "national_id":
            payload[k]["is_hash"] = True
    return payload

def build_backend_save_policy_v11(extraction_status, common, common_validation, row_validation, doc_type, is_image=False):
    reason_codes = []
    route = "save_to_database"
    backend_save_allowed = True

    if extraction_status in ["poor_quality_rejected", "ocr_failed", "ocr_unreliable_needs_reupload"]:
        backend_save_allowed = False
        route = "ask_user_reupload"
        reason_codes.append(extraction_status)

    if extraction_status == "unrelated_or_not_medical":
        backend_save_allowed = False
        route = "reject_unrelated"
        reason_codes.append("unrelated_or_not_medical")

    if extraction_status in ["partial_extraction", "needs_manual_review"]:
        backend_save_allowed = False
        route = "manual_review"
        reason_codes.append(extraction_status)

    if not common_validation["critical_fields_valid_for_backend"]:
        backend_save_allowed = False
        if route == "save_to_database":
            route = "manual_review"
        reason_codes.extend(common_validation["critical_field_reason_codes"])

    if row_validation["safe_row_count"] == 0 and doc_type == "lab":
        backend_save_allowed = False
        if route == "save_to_database":
            route = "manual_review"
        reason_codes.append("no_safe_lab_rows")

    if row_validation["unsafe_row_count"] > 0:
        reason_codes.append("some_rows_require_review")

    # Product/business safety: image rows are review-only unless everything passes.
    if is_image and backend_save_allowed:
        # Extra guard: image must have valid common fields and zero unsafe rows.
        if row_validation["unsafe_row_count"] > 0 or not common_validation["critical_fields_valid_for_backend"]:
            backend_save_allowed = False
            route = "manual_review"
            reason_codes.append("image_requires_review_due_to_field_or_row_uncertainty")

    common_payload = valid_common_payload_v11(common, common_validation["field_validation_status"])
    safe_payload = {
        "common_fields": common_payload,
        "lab_results": row_validation["safe_rows"],
    }

    if not backend_save_allowed:
        safe_payload = {"common_fields": {}, "lab_results": []}

    return {
        "backend_save_allowed": backend_save_allowed,
        "review_required": route == "manual_review",
        "reupload_required": route == "ask_user_reupload",
        "route": route,
        "reason_codes": sorted(set(reason_codes)),
        "safe_payload": safe_payload,
    }

def validate_rows_for_backend_v11(labs):
    counts = Counter()
    column_counts = Counter()
    unsafe = []
    safe = []
    backend_safe = []

    for r in labs:
        status = r.get("validation_status", "review")
        counts[status] += 1
        for k, v in (r.get("column_statuses") or {}).items():
            column_counts[f"{k}:{v}"] += 1

        if r.get("save_allowed"):
            safe.append(r)
        else:
            unsafe.append({
                "test_name_standard": r.get("test_name_standard"),
                "result_value": r.get("result_value"),
                "validation_status": status,
                "column_statuses": r.get("column_statuses"),
                "reason_codes": r.get("reason_codes", []),
                "source_text": r.get("source_text"),
            })

    return {
        "row_validation_status": dict(counts),
        "column_status_counts": dict(column_counts),
        "safe_row_count": len(safe),
        "unsafe_row_count": len(unsafe),
        "unsafe_rows_sample": unsafe[:100],
        "safe_rows": safe,
    }

def lab_to_parts_v11(labs):
    rows = []
    for r in labs:
        column_statuses = r.get("column_statuses") or column_statuses_for_row_v11(r)
        rows.append({
            "part_type": "lab_result",
            "section": r.get("section"),
            "field_name": r.get("test_name_standard"),
            "value": r.get("result_value"),
            "result_numeric": r.get("result_numeric"),
            "unit": r.get("unit"),
            "reference_range": r.get("reference_range"),
            "flag": r.get("flag"),
            "method": r.get("method"),
            "confidence": r.get("confidence"),
            "source_text": r.get("source_text"),
            "test_name_raw": r.get("test_name_raw"),
            "extraction_mode": r.get("extraction_mode"),
            "row_validation_status": r.get("validation_status"),
            "row_save_allowed": r.get("save_allowed"),
            "backend_row_save_allowed": r.get("backend_row_save_allowed"),
            "row_reason_codes": "|".join(r.get("reason_codes", [])),
            "backend_reason_codes": "|".join(r.get("backend_reason_codes", r.get("reason_codes", []))),
            "corrected_numeric": r.get("corrected_numeric"),
            "corrected_value": r.get("corrected_value"),
            "column_statuses_json": json.dumps(column_statuses, ensure_ascii=False),
            "test_name_status": column_statuses.get("test_name_status"),
            "result_status": column_statuses.get("result_status"),
            "unit_status": column_statuses.get("unit_status"),
            "reference_range_status": column_statuses.get("reference_range_status"),
            "flag_status": column_statuses.get("flag_status"),
            "method_status": column_statuses.get("method_status"),
        })
    return rows

print("v11 overrides loaded: doctor safety, backend_row_save_allowed, unsafe OCR enforcement, expected units, stricter tracking/culture.")


v11 overrides loaded: doctor safety, backend_row_save_allowed, unsafe OCR enforcement, expected units, stricter tracking/culture.


In [14]:

# =========================
# v12: Best extraction overrides before FastAPI integration
# =========================

# v12 mindset:
# - We return extraction + statuses. We do not "save".
# - For good images missing patient/date, rows can still be extracted and returned for review/context merge.
# - Persistence recommendation is advisory only.

REGION_DIR = DEBUG_DIR / "regions"
REGION_DIR.mkdir(parents=True, exist_ok=True)

# ---- More aliases for the provided samples ----
LAB_ALIASES.update({
    "DHT": ["DHT", "Dihydrotestosterone"],
    "LH": ["LH", "L.H"],
    "FSH": ["FSH", "F.S.H"],
    "Prolactin": ["Prolactin"],
    "Testosterone": ["Testosterone"],
    "Free Testosterone": ["Free Testosterone"],
    "Estradiol": ["Estradiol"],
    "DHEA-SO4": ["DHEA-SO4", "DHEA SO4", "DHEA-S04"],
    "17OH-Progesterone": ["17OH-Progesterone", "17OH Progesterone", "17-OH Progesterone"],
    "Free T3": ["Free T3", "FT3"],
    "Free T4": ["Free T4", "FT4"],
    "T3": ["T3", "Tri iodothyronine", "Triiodothyronine"],
    "T4": ["T4", "Thyroxine"],
    "Calcium": ["Calcium", "Ca"],
    "Phosphorus": ["Phosphorus", "P"],
    "Iron": ["Iron", "Fe"],
    "TIBC": ["TIBC", "Total Iron Binding Capacity"],
    "Bilirubin Total": ["Bilirubin Total", "Bilirubin(Total)"],
    "Bilirubin Direct": ["Bilirubin Direct", "Bilirubin(Direct)"],
    "Alkaline Phosphatase": ["Alkaline Phosphatase", "ALP"],
    "Zinc": ["Zinc"],
    "Folic Acid": ["Folic Acid", "Folate"],
    "Vitamin B12": ["Vitamin B12", "B12"],
    "Lymph Count": ["Lymph.Count", "Lymph Count", "LymphCount"],
})

ALL_ALIAS_PHRASES = sorted(
    [(std, alias) for std, aliases in LAB_ALIASES.items() for alias in aliases],
    key=lambda x: len(x[1]),
    reverse=True,
)
ALL_ALIAS_LOWER = [(std, alias, alias.lower()) for std, alias in ALL_ALIAS_PHRASES]

ROW_SANITY_LIMITS.update({
    "DHT": (1.0, 5000.0),
    "LH": (0.01, 300.0),
    "FSH": (0.01, 300.0),
    "Prolactin": (0.1, 500.0),
    "Testosterone": (0.01, 2000.0),
    "Free Testosterone": (0.01, 300.0),
    "Estradiol": (1.0, 5000.0),
    "DHEA-SO4": (1.0, 10000.0),
    "17OH-Progesterone": (0.01, 100.0),
    "T3": (0.1, 10.0),
    "T4": (0.1, 30.0),
    "Free T3": (0.1, 20.0),
    "Free T4": (0.1, 10.0),
    "Calcium": (1.0, 20.0),
    "Phosphorus": (0.5, 20.0),
    "Iron": (1.0, 500.0),
    "TIBC": (50.0, 800.0),
    "Bilirubin Total": (0.0, 50.0),
    "Bilirubin Direct": (0.0, 50.0),
    "Alkaline Phosphatase": (1.0, 2000.0),
    "Zinc": (1.0, 500.0),
    "Folic Acid": (0.1, 100.0),
    "Vitamin B12": (10.0, 5000.0),
    "Lymph Count": (0.1, 100.0),
})

EXPECTED_UNITS.update({
    "DHT": {"ng/dL", "ng/dl"},
    "LH": {"IU/L", "mIU/ml", "mIU/mL"},
    "FSH": {"mIU/ml", "mIU/mL", "IU/L"},
    "Prolactin": {"ng/ml", "ng/mL"},
    "Testosterone": {"ng/ml", "ng/mL", "ng/dL", "ng/dl"},
    "Free Testosterone": {"pg/ml", "pg/mL", "ng/dL", "ng/dl"},
    "Estradiol": {"pg/ml", "pg/mL"},
    "DHEA-SO4": {"ng/ml", "ng/mL", "µg/dL", "ug/dL"},
    "17OH-Progesterone": {"ng/ml", "ng/mL"},
    "T3": {"ng/ml", "ng/mL"},
    "T4": {"µg/dL", "ug/dL"},
    "Free T3": {"pg/ml", "pg/mL"},
    "Free T4": {"ng/dL", "ng/dl"},
    "Calcium": {"mg/dL", "mg/dl"},
    "Phosphorus": {"mg/dL", "mg/dl"},
    "Iron": {"µg/dL", "ug/dL", "mcg/dL"},
    "TIBC": {"µg/dL", "ug/dL", "mcg/dL"},
    "Bilirubin Total": {"mg/dL", "mg/dl"},
    "Bilirubin Direct": {"mg/dL", "mg/dl"},
    "Alkaline Phosphatase": {"U/L", "IU/L"},
    "Zinc": {"µg/dL", "ug/dL"},
    "Folic Acid": {"ng/ml", "ng/mL"},
    "Vitamin B12": {"pg/ml", "pg/mL"},
    "Lymph Count": {"1000/uL", "1000/µL", "10^3/uL", "10^3/µL"},
})

# ---- Page role / template detection ----
def classify_page_role_and_template(path, quality, text, lines):
    t = normalize_text(text)
    low = t.lower()
    fname = Path(path).name.lower()
    has_patient = bool(re.search(r"نام\s*بیمار|نام\s*مراجعه|patient\s*name", t, re.I))
    has_date = bool(re.search(r"تاریخ\s*پذیرش|report\s*date|date", t, re.I))
    has_section = bool(re.search(r"Hematology|Biochemistry|Hormone|Urine|Bacteriology|Serology|Special Biochemistry|Thyroid|Endocrinology", t, re.I))
    has_lab_table = bool(alias_hits(t)) or has_section
    has_culture = culture_signal(t)
    has_many_rows = len([l for l in lines if alias_hits(l.get("text",""))]) >= 3

    w = quality.get("width") or 0
    h = quality.get("height") or 0
    ratio = w / h if h else None

    if "thermal" in fname or (ratio and (ratio > 1.2 or ratio < 0.55) and re.search(r"\bWBC\b|\bRBC\b|\bPLT\b|\bHGB\b", t, re.I)):
        template = "thermal_or_receipt_cbc"
    elif re.search(r"Nobin|نوبین|Specialized and|Subspecialized", t, re.I):
        template = "nobin_photo_with_history_charts"
    elif re.search(r"آتابای|Atabay|Hormone|Urine Analysis|Bacteriology", t, re.I):
        template = "atabay_rotated_or_screenshot"
    elif re.search(r"درمانگاه\s*وحدت|Vahdat|وحدت", t, re.I):
        template = "vahdat_multi_page_photo"
    elif re.search(r"Emrooz|امروز", t, re.I):
        template = "emrooz_clean_screenshot"
    else:
        template = "generic_lab"

    if has_patient and has_date and has_lab_table:
        role = "standalone_report_page"
    elif has_lab_table and not (has_patient and has_date):
        role = "continuation_or_body_only_page"
    elif has_culture:
        role = "culture_result_page"
    elif has_patient and not has_lab_table:
        role = "header_or_demographic_page"
    else:
        role = "unknown_page_role"

    return {
        "page_role": role,
        "template_type": template,
        "has_patient_header": has_patient,
        "has_report_date_signal": has_date,
        "has_lab_table_signal": has_lab_table,
        "has_many_table_rows": has_many_rows,
        "has_culture_signal": has_culture,
        "requires_backend_context_for_save": bool(has_lab_table and not (has_patient and has_date)),
    }

# ---- Better image preprocessing: document crop + border crop + adaptive candidates ----
def crop_outer_whitespace_pil(img, margin=12):
    try:
        gray = np.array(img.convert("L"))
        mask = gray < 245
        ys, xs = np.where(mask)
        if len(xs) < 100:
            return img
        x1, x2 = max(0, xs.min()-margin), min(img.width, xs.max()+margin)
        y1, y2 = max(0, ys.min()-margin), min(img.height, ys.max()+margin)
        if (x2-x1) < img.width*0.35 or (y2-y1) < img.height*0.35:
            return img
        return img.crop((x1, y1, x2, y2))
    except Exception:
        return img

def document_contour_crop_pil(img):
    if not CV2_AVAILABLE:
        return img
    try:
        arr = np.array(img.convert("RGB"))
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (5,5), 0)
        edges = cv2.Canny(blur, 50, 150)
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return img
        c = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(c)
        if area < arr.shape[0] * arr.shape[1] * 0.20:
            return img
        x, y, w, h = cv2.boundingRect(c)
        pad = 12
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(arr.shape[1], x+w+pad), min(arr.shape[0], y+h+pad)
        if (x2-x1) < arr.shape[1]*0.35 or (y2-y1) < arr.shape[0]*0.35:
            return img
        return Image.fromarray(arr[y1:y2, x1:x2]).convert("RGB")
    except Exception:
        return img

def split_page_regions(img, sid):
    # Coarse regions: full, header, upper body, lower body.
    out = []
    W, H = img.size
    crops = {
        "full": (0, 0, W, H),
        "header_top_25": (0, 0, W, int(H*0.28)),
        "body_20_75": (0, int(H*0.18), W, int(H*0.78)),
        "lower_55_100": (0, int(H*0.55), W, H),
    }
    for name, box in crops.items():
        c = img.crop(box)
        if c.width * c.height > 10000:
            out.append((name, c))
    return out

def image_preprocess_variants(path, sid):
    base = ImageOps.exif_transpose(Image.open(path)).convert("RGB")

    doc_crop = document_contour_crop_pil(base)
    white_crop = crop_outer_whitespace_pil(base)
    doc_white = crop_outer_whitespace_pil(doc_crop)

    base_variants = [
        ("original_oriented", base),
        ("document_contour_crop", doc_crop),
        ("outer_whitespace_crop", white_crop),
        ("document_then_whitespace_crop", doc_white),
    ]

    variants = []
    def add(name, im):
        if im.width < 40 or im.height < 40:
            return
        variants.append((name, im.convert("RGB")))

    for base_name, im in base_variants:
        add(base_name, im)
        add(f"{base_name}_upscale2", im.resize((im.width*2, im.height*2), Image.Resampling.LANCZOS))
        gray = ImageOps.grayscale(im)
        add(f"{base_name}_gray_autocontrast", ImageOps.autocontrast(gray))
        add(f"{base_name}_mild_contrast", ImageEnhance.Contrast(gray).enhance(1.45))
        add(f"{base_name}_sharpen_light", ImageOps.autocontrast(gray.filter(ImageFilter.SHARPEN)))
        add(f"{base_name}_clahe_light", pil_clahe(im, 1.5))
        if CV2_AVAILABLE:
            arr = np.array(gray)
            thr = cv2.adaptiveThreshold(arr,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,11)
            add(f"{base_name}_threshold_fallback", Image.fromarray(thr))

    # Add rotations for full/cropped main variants.
    if CONFIG["run_rotation_variants"]:
        rotate_source = [(n, im) for n, im in variants if n in {
            "original_oriented", "outer_whitespace_crop", "document_then_whitespace_crop",
            "outer_whitespace_crop_upscale2", "document_then_whitespace_crop_upscale2",
            "outer_whitespace_crop_gray_autocontrast", "document_then_whitespace_crop_gray_autocontrast"
        }]
        for name, im in rotate_source:
            for deg in [90, 180, 270]:
                add(f"{name}_rot{deg}", im.rotate(deg, expand=True))

    # Region crops are useful for page-body-only screenshots and multi-section pages.
    region_source = doc_white if doc_white.width * doc_white.height > base.width * base.height * 0.25 else base
    for region_name, region_img in split_page_regions(region_source, sid):
        if region_name != "full":
            add(f"region_{region_name}", region_img)
            add(f"region_{region_name}_upscale2", region_img.resize((region_img.width*2, region_img.height*2), Image.Resampling.LANCZOS))
            add(f"region_{region_name}_gray_autocontrast", ImageOps.autocontrast(ImageOps.grayscale(region_img)))

    # De-duplicate by name + size.
    seen = set()
    deduped = []
    for n, im in variants:
        key = (n, im.size)
        if key not in seen:
            seen.add(key)
            deduped.append((n, im))
    variants = deduped

    if CONFIG["save_variant_images"]:
        folder = VARIANT_IMAGE_DIR / sid
        folder.mkdir(parents=True, exist_ok=True)
        for name, im in variants[:120]:
            try:
                im.save(folder / f"{name}.png")
            except Exception:
                pass

    return variants

# ---- Better OCR candidate scoring: do not mistake review-required as failure ----
def select_best_candidate_v12(candidates, quality):
    if not candidates:
        return OCRCandidate("empty", 1, "none", None, "none", "", 0, 0, "empty_text", {}, [], [], "no_candidates")

    evaluated = []
    for c in candidates:
        text = normalize_text(c.text)
        common = extract_common_fields_v11(text)
        labs = extract_lab_results_v11(text, c.lines, unsafe_ocr_context=False)
        row_val = validate_rows_for_backend_v11(labs)
        doc_type, _, _ = classify_document(text)
        page_ctx = classify_page_role_and_template("unknown", quality, text, c.lines)

        safe_rows = row_val["safe_row_count"]
        unsafe_rows = row_val["unsafe_row_count"]
        total_rows = safe_rows + unsafe_rows

        # v12 rewards extraction yield, but separately from saveability.
        extraction_yield_bonus = min(0.18, len(labs) * 0.012)
        safe_row_bonus = min(0.12, safe_rows * 0.010)

        common_bonus = 0.0
        if common.get("date_of_test_or_report", {}).get("value"):
            common_bonus += 0.03
        if common.get("patient_name", {}).get("value") or common.get("national_id", {}).get("value"):
            common_bonus += 0.03

        continuation_bonus = 0.04 if page_ctx["page_role"] in {"continuation_or_body_only_page", "culture_result_page"} and len(labs) > 0 else 0.0
        culture_bonus = 0.08 if page_ctx["has_culture_signal"] else 0.0

        invalid_ratio_penalty = 0.0
        if total_rows:
            invalid_ratio_penalty = min(0.12, unsafe_rows / total_rows * 0.08)

        vertical_pen = vertical_line_penalty(c.lines)
        suspicious_pen = suspicious_common_penalty(common)

        no_text_penalty = 0.15 if len(text) < 40 else 0.0

        v12_score = (
            float(c.score)
            + extraction_yield_bonus
            + safe_row_bonus
            + common_bonus
            + continuation_bonus
            + culture_bonus
            - invalid_ratio_penalty
            - vertical_pen
            - suspicious_pen
            - no_text_penalty
        )
        v12_score = max(0.0, min(1.0, v12_score))

        c.score_details = dict(c.score_details)
        c.score_details.update({
            "v12_selection_score": round(v12_score, 3),
            "v12_page_role": page_ctx["page_role"],
            "v12_template_type": page_ctx["template_type"],
            "v12_extracted_row_count": len(labs),
            "v12_safe_row_count": safe_rows,
            "v12_unsafe_row_count": unsafe_rows,
            "v12_extraction_yield_bonus": round(extraction_yield_bonus, 3),
            "v12_safe_row_bonus": round(safe_row_bonus, 3),
            "v12_common_bonus": round(common_bonus, 3),
            "v12_continuation_bonus": round(continuation_bonus, 3),
            "v12_culture_bonus": round(culture_bonus, 3),
            "v12_invalid_ratio_penalty": round(invalid_ratio_penalty, 3),
            "v12_vertical_line_penalty": round(vertical_pen, 3),
            "v12_suspicious_common_penalty": round(suspicious_pen, 3),
        })
        evaluated.append((v12_score, c.confidence, len(text), c))

    return sorted(evaluated, key=lambda x: (x[0], x[1], x[2]), reverse=True)[0][3]

# ---- Better status language for API: not save decision, only recommendation ----
def decide_extraction_status_v12(path, quality, chosen, medical_ok, doc_type, common, labs, page_ctx):
    if quality.get("status") in ["poor_quality", "unreadable"] and Path(path).suffix.lower() != ".pdf":
        return "poor_quality_rejected", ["input_quality_too_poor"]

    if not chosen.text.strip():
        return "ocr_failed", ["ocr_failed"]

    if chosen.status in ["poor_ocr_text", "gibberish_or_bad_layout_text"] and not page_ctx.get("has_culture_signal"):
        return "ocr_unreliable_needs_reupload", [f"ocr_layout_status={chosen.status}"]

    if not medical_ok and not labs:
        return "unrelated_or_not_medical", ["no_medical_signals"]

    if doc_type == "lab" or labs:
        if len(labs) == 0:
            return "extracted_no_rows", ["medical_document_but_no_structured_rows"]
        if page_ctx["requires_backend_context_for_save"]:
            return "extracted_requires_context", ["missing_patient_or_date_on_this_page"]
        return "extracted_good", []

    return "extracted_review_required", ["unknown_or_weak_document_type"]

def persistence_recommendation_v12(extraction_status, common, common_validation, row_validation, doc_type, is_image, page_ctx):
    # This is not an action. Backend can override.
    reason_codes = []
    recommended_action = "save_candidate"
    recommended_save = True

    if extraction_status in ["poor_quality_rejected", "ocr_failed", "ocr_unreliable_needs_reupload"]:
        recommended_save = False
        recommended_action = "reupload_recommended"
        reason_codes.append(extraction_status)

    if extraction_status == "unrelated_or_not_medical":
        recommended_save = False
        recommended_action = "reject_or_ignore"
        reason_codes.append("unrelated_or_not_medical")

    if extraction_status in ["extracted_requires_context", "extracted_review_required", "extracted_no_rows"]:
        recommended_save = False
        recommended_action = "manual_review"
        reason_codes.append(extraction_status)

    if not common_validation["critical_fields_valid_for_backend"]:
        recommended_save = False
        if recommended_action == "save_candidate":
            recommended_action = "manual_review"
        reason_codes.extend(common_validation["critical_field_reason_codes"])

    if row_validation["safe_row_count"] == 0 and doc_type == "lab":
        recommended_save = False
        if recommended_action == "save_candidate":
            recommended_action = "manual_review"
        reason_codes.append("no_safe_lab_rows")

    if row_validation["unsafe_row_count"] > 0:
        reason_codes.append("some_rows_require_review")

    if page_ctx.get("requires_backend_context_for_save"):
        recommended_save = False
        if recommended_action == "save_candidate":
            recommended_action = "manual_review"
        reason_codes.append("requires_backend_patient_date_context")

    if is_image and recommended_save:
        # image auto-save requires all critical fields and no unsafe rows
        if row_validation["unsafe_row_count"] > 0 or not common_validation["critical_fields_valid_for_backend"]:
            recommended_save = False
            recommended_action = "manual_review"
            reason_codes.append("image_requires_review_due_to_field_or_row_uncertainty")

    safe_common = valid_common_payload_v11(common, common_validation["field_validation_status"])
    safe_rows = row_validation["safe_rows"]

    safe_payload_candidate = {"common_fields": safe_common, "lab_results": safe_rows} if recommended_save else {"common_fields": {}, "lab_results": []}
    review_payload = {
        "common_fields": common,
        "field_validation": common_validation,
        "lab_results": row_validation.get("all_rows_for_review", []),
        "page_context": page_ctx,
    }

    return {
        "recommended_action": recommended_action,
        "recommended_save": recommended_save,
        "review_required": recommended_action == "manual_review",
        "reupload_required": recommended_action == "reupload_recommended",
        "reason_codes": sorted(set(reason_codes)),
        "safe_payload_candidate": safe_payload_candidate,
        "review_payload": review_payload,
    }

def add_backend_row_flags_v12(rows, recommended_save):
    out = []
    for r in rows:
        r = dict(r)
        # row_save_allowed = internally valid; backend_row_save_recommendation = doc + row
        r["backend_row_save_recommendation"] = bool(recommended_save and r.get("save_allowed"))
        r["backend_row_save_allowed"] = r["backend_row_save_recommendation"]  # compatibility
        if not r["backend_row_save_recommendation"]:
            reasons = set(r.get("reason_codes", []))
            if not recommended_save:
                reasons.add("document_not_recommended_for_save")
            r["backend_reason_codes"] = sorted(reasons)
        else:
            r["backend_reason_codes"] = r.get("reason_codes", [])
        return_row = r
        out.append(return_row)
    return out

def field_backend_usable_map(common, common_validation, recommended_save):
    out = {}
    for k, v in common.items():
        st = common_validation["field_validation_status"].get(k, "missing")
        usable = bool(recommended_save and st == "valid")
        if k == "doctor_name" and not v.get("explicit_doctor_label"):
            usable = False
        out[k] = usable
    return out

def common_to_parts_v12(common, field_validation_status, field_backend_usable):
    rows = []
    for k, v in common.items():
        rows.append({
            "part_type": "common_field",
            "section": "header",
            "field_name": k,
            "value": v.get("value"),
            "confidence": v.get("confidence", 0),
            "source_text": v.get("source_text"),
            "field_validation_status": field_validation_status.get(k, "unknown"),
            "field_backend_usable": field_backend_usable.get(k, False),
        })
    return rows

def lab_to_parts_v12(labs):
    rows = []
    for r in labs:
        column_statuses = r.get("column_statuses") or column_statuses_for_row_v11(r)
        rows.append({
            "part_type": "lab_result",
            "section": r.get("section"),
            "field_name": r.get("test_name_standard"),
            "value": r.get("result_value"),
            "result_numeric": r.get("result_numeric"),
            "unit": r.get("unit"),
            "reference_range": r.get("reference_range"),
            "flag": r.get("flag"),
            "method": r.get("method"),
            "confidence": r.get("confidence"),
            "source_text": r.get("source_text"),
            "test_name_raw": r.get("test_name_raw"),
            "extraction_mode": r.get("extraction_mode"),
            "row_validation_status": r.get("validation_status"),
            "row_save_allowed": r.get("save_allowed"),
            "backend_row_save_recommendation": r.get("backend_row_save_recommendation"),
            "backend_row_save_allowed": r.get("backend_row_save_allowed"),
            "row_reason_codes": "|".join(r.get("reason_codes", [])),
            "backend_reason_codes": "|".join(r.get("backend_reason_codes", r.get("reason_codes", []))),
            "corrected_numeric": r.get("corrected_numeric"),
            "corrected_value": r.get("corrected_value"),
            "column_statuses_json": json.dumps(column_statuses, ensure_ascii=False),
            "test_name_status": column_statuses.get("test_name_status"),
            "result_status": column_statuses.get("result_status"),
            "unit_status": column_statuses.get("unit_status"),
            "reference_range_status": column_statuses.get("reference_range_status"),
            "flag_status": column_statuses.get("flag_status"),
            "method_status": column_statuses.get("method_status"),
        })
    return rows

print("v12 overrides loaded: page roles, better image preprocessing, extraction-first statuses, persistence recommendation.")


v12 overrides loaded: page roles, better image preprocessing, extraction-first statuses, persistence recommendation.


In [15]:

# =========================
# v12.1 Hotfix: page-context and save recommendation
# =========================

def common_has_patient_identity(common):
    return bool(
        common.get("patient_name", {}).get("value")
        or common.get("national_id", {}).get("value")
    )

def common_has_valid_report_date(common):
    value = common.get("date_of_test_or_report", {}).get("value")
    return bool(value and validate_jalali_date(str(value)))

def classify_page_role_and_template(path, quality, text, lines):
    # v12.1: use extracted common fields as evidence, not only raw labels.
    t = normalize_text(text)
    low = t.lower()
    common_probe = extract_common_fields_v11(t)

    raw_patient_signal = bool(re.search(r"نام\s*بیمار|نام\s*مراجعه|patient\s*name", t, re.I))
    raw_date_signal = bool(re.search(r"تاریخ\s*پذیرش|تاریخ\s*گزارش|report\s*date|date", t, re.I))

    extracted_patient_signal = common_has_patient_identity(common_probe)
    extracted_date_signal = common_has_valid_report_date(common_probe)

    # TAV PDF has a compact header without "نام بیمار" label.
    tav_compact_patient_signal = bool(re.search(r"-\s*(?:آقای|آقاي|خانم)\s+[^\n]{1,80}?-\s*دکتر\s*[0-9]{1,3}\s*[:：]\s*سن", t))

    has_patient = raw_patient_signal or extracted_patient_signal or tav_compact_patient_signal
    has_date = raw_date_signal or extracted_date_signal

    has_section = bool(re.search(r"Hematology|Biochemistry|Hormone|Urine|Bacteriology|Serology|Special Biochemistry|Thyroid|Endocrinology|Coagulation", t, re.I))
    has_lab_table = bool(alias_hits(t)) or has_section
    has_culture = culture_signal(t)
    has_many_rows = len([l for l in lines if alias_hits(l.get("text", ""))]) >= 3

    w = quality.get("width") or 0
    h = quality.get("height") or 0
    ratio = w / h if h else None

    if "thermal" in str(path).lower() or (ratio and (ratio > 1.2 or ratio < 0.55) and re.search(r"\bWBC\b|\bRBC\b|\bPLT\b|\bHGB\b", t, re.I)):
        template = "thermal_or_receipt_cbc"
    elif re.search(r"Nobin|نوبین|Specialized and|Subspecialized", t, re.I):
        template = "nobin_photo_with_history_charts"
    elif re.search(r"آتابای|Atabay|Hormone|Urine Analysis|Bacteriology", t, re.I):
        template = "atabay_rotated_or_screenshot"
    elif re.search(r"درمانگاه\s*وحدت|Vahdat|وحدت", t, re.I):
        template = "vahdat_multi_page_photo"
    elif re.search(r"Emrooz|امروز", t, re.I):
        template = "emrooz_clean_screenshot"
    elif re.search(r"آزمایشگاه\s*پاتوبیولوژی\s*تاو|TAV|تاو", t, re.I):
        template = "tav_text_pdf"
    else:
        template = "generic_lab"

    if has_patient and has_date and has_lab_table:
        role = "standalone_report_page"
    elif has_lab_table and not (has_patient and has_date):
        role = "continuation_or_body_only_page"
    elif has_culture:
        role = "culture_result_page"
    elif has_patient and not has_lab_table:
        role = "header_or_demographic_page"
    else:
        role = "unknown_page_role"

    return {
        "page_role": role,
        "template_type": template,
        "has_patient_header": has_patient,
        "has_report_date_signal": has_date,
        "has_lab_table_signal": has_lab_table,
        "has_many_table_rows": has_many_rows,
        "has_culture_signal": has_culture,
        "requires_backend_context_for_save": bool(has_lab_table and not (has_patient and has_date)),
        "page_context_evidence": {
            "raw_patient_signal": raw_patient_signal,
            "raw_date_signal": raw_date_signal,
            "extracted_patient_signal": extracted_patient_signal,
            "extracted_date_signal": extracted_date_signal,
            "tav_compact_patient_signal": tav_compact_patient_signal,
        },
    }

def decide_extraction_status_v12(path, quality, chosen, medical_ok, doc_type, common, labs, page_ctx):
    if quality.get("status") in ["poor_quality", "unreadable"] and Path(path).suffix.lower() != ".pdf":
        return "poor_quality_rejected", ["input_quality_too_poor"]

    if not chosen.text.strip():
        return "ocr_failed", ["ocr_failed"]

    if chosen.status in ["poor_ocr_text", "gibberish_or_bad_layout_text"] and not page_ctx.get("has_culture_signal"):
        return "ocr_unreliable_needs_reupload", [f"ocr_layout_status={chosen.status}"]

    if not medical_ok and not labs:
        return "unrelated_or_not_medical", ["no_medical_signals"]

    if doc_type == "lab" or labs:
        if len(labs) == 0:
            return "extracted_no_rows", ["medical_document_but_no_structured_rows"]
        # v12.1: only require context if extracted common fields do not already prove patient/date.
        has_context = common_has_patient_identity(common) and common_has_valid_report_date(common)
        if page_ctx["requires_backend_context_for_save"] and not has_context:
            return "extracted_requires_context", ["missing_patient_or_date_on_this_page"]
        return "extracted_good", []

    return "extracted_review_required", ["unknown_or_weak_document_type"]

def persistence_recommendation_v12(extraction_status, common, common_validation, row_validation, doc_type, is_image, page_ctx):
    reason_codes = []
    recommended_action = "save_candidate"
    recommended_save = True

    if extraction_status in ["poor_quality_rejected", "ocr_failed", "ocr_unreliable_needs_reupload"]:
        recommended_save = False
        recommended_action = "reupload_recommended"
        reason_codes.append(extraction_status)

    if extraction_status == "unrelated_or_not_medical":
        recommended_save = False
        recommended_action = "reject_or_ignore"
        reason_codes.append("unrelated_or_not_medical")

    if extraction_status in ["extracted_requires_context", "extracted_review_required", "extracted_no_rows"]:
        recommended_save = False
        recommended_action = "manual_review"
        reason_codes.append(extraction_status)

    if not common_validation["critical_fields_valid_for_backend"]:
        recommended_save = False
        if recommended_action == "save_candidate":
            recommended_action = "manual_review"
        reason_codes.extend(common_validation["critical_field_reason_codes"])

    if row_validation["safe_row_count"] == 0 and doc_type == "lab":
        recommended_save = False
        if recommended_action == "save_candidate":
            recommended_action = "manual_review"
        reason_codes.append("no_safe_lab_rows")

    if row_validation["unsafe_row_count"] > 0:
        reason_codes.append("some_rows_require_review")

    # v12.1: do not block when common validation already has context.
    if page_ctx.get("requires_backend_context_for_save") and not common_validation["critical_fields_valid_for_backend"]:
        recommended_save = False
        if recommended_action == "save_candidate":
            recommended_action = "manual_review"
        reason_codes.append("requires_backend_patient_date_context")

    if is_image and recommended_save:
        # image auto-save still requires very strong evidence
        if row_validation["unsafe_row_count"] > 0 or not common_validation["critical_fields_valid_for_backend"]:
            recommended_save = False
            recommended_action = "manual_review"
            reason_codes.append("image_requires_review_due_to_field_or_row_uncertainty")

    safe_common = valid_common_payload_v11(common, common_validation["field_validation_status"])
    safe_rows = row_validation["safe_rows"]

    safe_payload_candidate = {"common_fields": safe_common, "lab_results": safe_rows} if recommended_save else {"common_fields": {}, "lab_results": []}
    review_payload = {
        "common_fields": common,
        "field_validation": common_validation,
        "lab_results": row_validation.get("all_rows_for_review", []),
        "page_context": page_ctx,
    }

    return {
        "recommended_action": recommended_action,
        "recommended_save": recommended_save,
        "review_required": recommended_action == "manual_review",
        "reupload_required": recommended_action == "reupload_recommended",
        "reason_codes": sorted(set(reason_codes)),
        "safe_payload_candidate": safe_payload_candidate,
        "review_payload": review_payload,
    }

print("v12.1 hotfix loaded: clean PDFs with extracted patient/date context should become save candidates.")


v12.1 hotfix loaded: clean PDFs with extracted patient/date context should become save candidates.


In [16]:

# =========================
# v12 Batch: extraction-first API-style output
# =========================

candidate_rows = []
words_rows = []
lines_rows = []
sections_rows = []
summary_rows = []
parts_rows = []
json_rows = []
text_rows = []
documents = []
safe_payload_candidates = []
review_payloads = []
reupload_recommended = []

for index, path in enumerate(FILES, start=1):
    sid = f"{index:04d}_{safe_id(path)}"
    print(f"[v12 {index}/{len(FILES)}] {path.name}")

    try:
        quality = assess_file_quality(path)
        source_preview = create_source_preview(path, sid)

        candidates, pdf_meta = get_ocr_candidates_for_file(path, sid)
        chosen = select_best_candidate_v12(candidates, quality)
        text = normalize_text(chosen.text)

        for c in candidates:
            row = {
                "index": index,
                "filename": path.name,
                "source_relative_path": relpath(path),
                "candidate_id": c.candidate_id,
                "page_number": c.page_number,
                "variant_name": c.variant_name,
                "psm": c.psm,
                "lang": c.lang,
                "confidence": c.confidence,
                "layout_score_v9": c.score,
                "layout_status": c.status,
                "text_length": len(c.text),
                "error": c.error,
                "is_chosen": c.candidate_id == chosen.candidate_id,
            }
            for k, v in c.score_details.items():
                row[f"score_{k}"] = v
            candidate_rows.append(row)

        wr, lr, sr = make_words_lines_sections_tables(index, path.name, chosen, relpath(path))
        words_rows.extend(wr)
        lines_rows.extend(lr)
        sections_rows.extend(sr)

        page_ctx = classify_page_role_and_template(path, quality, text, chosen.lines)
        medical_ok, med_hits, relevance_score = is_medical_text(text)
        doc_type, doc_conf, doc_signals = classify_document(text)
        if len(alias_hits(text)) >= 2 or page_ctx["has_lab_table_signal"]:
            doc_type = "lab"
            doc_conf = max(doc_conf, 0.75)

        common = extract_common_fields_v11(text)

        initial_unsafe_ocr = chosen.status in {"poor_ocr_text", "gibberish_or_bad_layout_text", "empty_text"}
        labs = extract_lab_results_v11(text, chosen.lines, unsafe_ocr_context=initial_unsafe_ocr)

        extraction_status, warnings = decide_extraction_status_v12(
            path, quality, chosen, medical_ok, doc_type, common, labs, page_ctx
        )

        final_unsafe_ocr = extraction_status in {"ocr_unreliable_needs_reupload", "ocr_failed", "poor_quality_rejected"} or initial_unsafe_ocr

        common_validation = validate_common_fields_v10(
            common,
            doc_type,
            chosen.status,
            CONFIG["backend_patient_context_available"],
        )

        if final_unsafe_ocr:
            common_validation = force_common_unsafe_context(common, common_validation)
            labs = force_rows_unsafe_ocr_context(labs)

        row_validation = validate_rows_for_backend_v11(labs)
        row_validation["all_rows_for_review"] = labs

        is_image = path.suffix.lower() != ".pdf"
        persistence = persistence_recommendation_v12(
            extraction_status,
            common,
            common_validation,
            row_validation,
            doc_type,
            is_image=is_image,
            page_ctx=page_ctx,
        )

        labs = add_backend_row_flags_v12(labs, persistence["recommended_save"])
        backend_safe_rows = [r for r in labs if r.get("backend_row_save_recommendation")]
        row_validation["backend_safe_row_count"] = len(backend_safe_rows)
        row_validation["backend_blocked_row_count"] = len(labs) - len(backend_safe_rows)
        if persistence["recommended_save"]:
            persistence["safe_payload_candidate"]["lab_results"] = backend_safe_rows
        else:
            persistence["safe_payload_candidate"] = {"common_fields": {}, "lab_results": []}

        field_backend_usable = field_backend_usable_map(common, common_validation, persistence["recommended_save"])
        parts = common_to_parts_v12(common, common_validation["field_validation_status"], field_backend_usable) + lab_to_parts_v12(labs)

        text_path = TEXT_DIR / f"{sid}.txt"
        sanitized_text_path = SANITIZED_TEXT_DIR / f"{sid}.txt"
        json_path = JSON_DIR / f"{sid}.json"
        text_path.write_text(text, encoding="utf-8")
        sanitized_text_path.write_text(sanitize_text(text), encoding="utf-8")

        doc_json = {
            "api_contract_note": "This response returns extraction results, statuses, scores, and evidence only. Backend decides persistence.",
            "document": {
                "index": index,
                "filename": path.name,
                "source_relative_path": relpath(path),
                "file_ext": path.suffix.lower(),
                "mime_type": file_mime_guess(path),
                "file_size_kb": round(path.stat().st_size / 1024, 2),
                "extraction_status": extraction_status,
                "warnings": warnings,
                "document_type": doc_type,
                "document_type_confidence": doc_conf,
                "relevance_score": relevance_score,
                "medical_hits": med_hits,
                "source_preview": source_preview,
                "is_image": is_image,
                "page_context": page_ctx,
            },
            "quality": quality,
            "ocr": {
                "chosen_candidate_id": chosen.candidate_id,
                "success": bool(text.strip()),
                "confidence": chosen.confidence,
                "text_length": len(text),
                "variant_name": chosen.variant_name,
                "psm": chosen.psm,
                "lang": chosen.lang,
                "layout_score_v9": chosen.score,
                "layout_status": chosen.status,
                "final_unsafe_ocr_context": final_unsafe_ocr,
                "score_details": chosen.score_details,
            },
            "persistence_recommendation": persistence,
            "common_fields": common,
            "field_validation": common_validation,
            "field_backend_usable": field_backend_usable,
            "row_validation": {k: v for k, v in row_validation.items() if k not in {"safe_rows", "all_rows_for_review"}},
            "safe_payload_candidate": persistence["safe_payload_candidate"],
            "review_payload": persistence["review_payload"],
            "lab_results": labs,
            "extracted_parts": parts,
            "extracted_text": text,
        }
        json_path.write_text(json.dumps(doc_json, ensure_ascii=False, indent=2), encoding="utf-8")
        documents.append(doc_json)

        if persistence["recommended_save"]:
            safe_payload_candidates.append(persistence["safe_payload_candidate"])
        if persistence["review_required"]:
            review_payloads.append(doc_json)
        if persistence["reupload_required"]:
            reupload_recommended.append(doc_json)

        for r in parts:
            rr = dict(r)
            rr.update({
                "index": index,
                "filename": path.name,
                "extraction_status": extraction_status,
                "recommended_action": persistence["recommended_action"],
                "recommended_save": persistence["recommended_save"],
                "document_type": doc_type,
                "source_relative_path": relpath(path),
                "text_relative_path": relpath(text_path),
                "json_relative_path": relpath(json_path),
                "page_role": page_ctx["page_role"],
                "template_type": page_ctx["template_type"],
            })
            parts_rows.append(rr)

        summary_rows.append({
            "index": index,
            "filename": path.name,
            "source_relative_path": relpath(path),
            "is_image": is_image,
            "page_role": page_ctx["page_role"],
            "template_type": page_ctx["template_type"],
            "requires_backend_context_for_save": page_ctx["requires_backend_context_for_save"],
            "extraction_status": extraction_status,
            "recommended_action": persistence["recommended_action"],
            "recommended_save": persistence["recommended_save"],
            "review_required": persistence["review_required"],
            "reupload_required": persistence["reupload_required"],
            "reason_codes": "|".join(persistence["reason_codes"]),
            "warnings": ";".join(warnings),
            "quality_status": quality.get("status"),
            "quality_score": quality.get("overall_quality_score"),
            "quality_issues": ";".join(quality.get("issues", [])),
            "ocr_success": bool(text.strip()),
            "ocr_confidence": round(float(chosen.confidence or 0), 3),
            "ocr_text_length": len(text),
            "ocr_layout_status": chosen.status,
            "final_unsafe_ocr_context": final_unsafe_ocr,
            "ocr_v12_selection_score": chosen.score_details.get("v12_selection_score"),
            "chosen_variant": chosen.variant_name,
            "chosen_psm": chosen.psm,
            "chosen_lang": chosen.lang,
            "document_type": doc_type,
            "document_type_confidence": doc_conf,
            "common_field_count": sum(1 for v in common.values() if v.get("value") is not None),
            "critical_fields_valid_for_backend": common_validation["critical_fields_valid_for_backend"],
            "field_validation_status": json.dumps(common_validation["field_validation_status"], ensure_ascii=False),
            "row_validation_status": json.dumps(row_validation["row_validation_status"], ensure_ascii=False),
            "column_status_counts": json.dumps(row_validation["column_status_counts"], ensure_ascii=False),
            "safe_row_count": row_validation["safe_row_count"],
            "unsafe_row_count": row_validation["unsafe_row_count"],
            "backend_safe_row_count": row_validation["backend_safe_row_count"],
            "backend_blocked_row_count": row_validation["backend_blocked_row_count"],
            "lab_result_count": len(labs),
            "patient_name_found": common.get("patient_name", {}).get("value") is not None,
            "date_found": common.get("date_of_test_or_report", {}).get("value") is not None,
            "age_found": common.get("age", {}).get("value") is not None,
            "sex_found": common.get("sex", {}).get("value") is not None,
            "national_id_hash_found": common.get("national_id", {}).get("value") is not None,
            "explicit_doctor_label": common.get("doctor_name", {}).get("explicit_doctor_label"),
            "text_relative_path": relpath(text_path),
            "json_relative_path": relpath(json_path),
            "source_preview": source_preview,
        })

        json_rows.append({
            "index": index,
            "filename": path.name,
            "extraction_status": extraction_status,
            "recommended_action": persistence["recommended_action"],
            "recommended_save": persistence["recommended_save"],
            "document_type": doc_type,
            "page_context_json": json.dumps(page_ctx, ensure_ascii=False),
            "quality_json": json.dumps(quality, ensure_ascii=False),
            "ocr_json": json.dumps(doc_json["ocr"], ensure_ascii=False),
            "common_fields_json": json.dumps(common, ensure_ascii=False),
            "field_validation_json": json.dumps(common_validation, ensure_ascii=False),
            "field_backend_usable_json": json.dumps(field_backend_usable, ensure_ascii=False),
            "row_validation_json": json.dumps({k: v for k, v in row_validation.items() if k not in {"safe_rows", "all_rows_for_review"}}, ensure_ascii=False),
            "persistence_recommendation_json": json.dumps(persistence, ensure_ascii=False),
            "safe_payload_candidate_json": json.dumps(doc_json["safe_payload_candidate"], ensure_ascii=False),
            "review_payload_json": json.dumps(doc_json["review_payload"], ensure_ascii=False),
            "lab_results_json": json.dumps(labs, ensure_ascii=False),
            "extracted_parts_json": json.dumps(parts, ensure_ascii=False),
            "document_json": json.dumps(doc_json, ensure_ascii=False),
            "source_relative_path": relpath(path),
            "text_relative_path": relpath(text_path),
            "json_relative_path": relpath(json_path),
        })

        text_rows.append({
            "index": index,
            "filename": path.name,
            "source_relative_path": relpath(path),
            "text_relative_path": relpath(text_path),
            "sanitized_text_relative_path": relpath(sanitized_text_path),
            "extracted_text": text,
            "sanitized_text": sanitize_text(text),
        })

    except Exception as e:
        traceback.print_exc()
        summary_rows.append({
            "index": index,
            "filename": path.name,
            "source_relative_path": relpath(path),
            "extraction_status": "notebook_error",
            "recommended_save": False,
            "recommended_action": "manual_review",
            "reason_codes": str(e),
        })

summary_df = pd.DataFrame(summary_rows)
parts_df = pd.DataFrame(parts_rows)
json_rows_df = pd.DataFrame(json_rows)
texts_df = pd.DataFrame(text_rows)
candidates_df = pd.DataFrame(candidate_rows)
words_df = pd.DataFrame(words_rows)
lines_df = pd.DataFrame(lines_rows)
sections_df = pd.DataFrame(sections_rows)

paths = {
    "document_summary.csv": OUTPUT_DIR / "document_summary.csv",
    "extracted_parts.csv": OUTPUT_DIR / "extracted_parts.csv",
    "document_json_rows.csv": OUTPUT_DIR / "document_json_rows.csv",
    "extracted_texts.csv": OUTPUT_DIR / "extracted_texts.csv",
    "ocr_candidates.csv": OUTPUT_DIR / "ocr_candidates.csv",
    "ocr_words.csv": OUTPUT_DIR / "ocr_words.csv",
    "ocr_lines.csv": OUTPUT_DIR / "ocr_lines.csv",
    "ocr_sections.csv": OUTPUT_DIR / "ocr_sections.csv",
    "documents.jsonl": OUTPUT_DIR / "documents.jsonl",
    "safe_payload_candidates.jsonl": OUTPUT_DIR / "safe_payload_candidates.jsonl",
    "review_payloads.jsonl": OUTPUT_DIR / "review_payloads.jsonl",
    "reupload_recommended.jsonl": OUTPUT_DIR / "reupload_recommended.jsonl",
}

summary_df.to_csv(paths["document_summary.csv"], index=False, encoding="utf-8-sig")
parts_df.to_csv(paths["extracted_parts.csv"], index=False, encoding="utf-8-sig")
json_rows_df.to_csv(paths["document_json_rows.csv"], index=False, encoding="utf-8-sig")
texts_df.to_csv(paths["extracted_texts.csv"], index=False, encoding="utf-8-sig")
candidates_df.to_csv(paths["ocr_candidates.csv"], index=False, encoding="utf-8-sig")
words_df.to_csv(paths["ocr_words.csv"], index=False, encoding="utf-8-sig")
lines_df.to_csv(paths["ocr_lines.csv"], index=False, encoding="utf-8-sig")
sections_df.to_csv(paths["ocr_sections.csv"], index=False, encoding="utf-8-sig")

for key, data in [
    ("documents.jsonl", documents),
    ("safe_payload_candidates.jsonl", safe_payload_candidates),
    ("review_payloads.jsonl", review_payloads),
    ("reupload_recommended.jsonl", reupload_recommended),
]:
    with paths[key].open("w", encoding="utf-8") as f:
        for d in data:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

print("Saved v12 outputs:")
for name, p in paths.items():
    print("-", relpath(p))

print("\nExtraction status counts:")
display(summary_df["extraction_status"].value_counts().to_frame("count"))

print("\nRecommended action counts:")
display(summary_df["recommended_action"].value_counts().to_frame("count"))

print("\nRecommended save:")
display(summary_df["recommended_save"].value_counts().to_frame("count"))

print("\nPage roles:")
display(summary_df["page_role"].value_counts().to_frame("count"))

display(summary_df)
display(parts_df.head(350))


[v12 1/20] -2147483648_-210193.jpg
[v12 2/20] -2147483648_-210195.jpg
[v12 3/20] 0014161672_14041209_O_00404121721.pdf
[v12 4/20] 0020139871_14041209_O_00404121714.pdf
[v12 5/20] 0021858845_14041209_O_00404121731.pdf
[v12 6/20] 0023471026_14041209_O_00404121728.pdf
[v12 7/20] 0024150010_14041209_O_00404121722.pdf
[v12 8/20] 0025631314_14041209_O_00404121726.pdf
[v12 9/20] 0025692283_14041209_O_00404121730.pdf
[v12 10/20] 20260427_181554.jpg
[v12 11/20] 20260427_181636.jpg
[v12 12/20] 20260427_181654.jpg
[v12 13/20] 20260427_181713.jpg
[v12 14/20] 20260427_181919.jpg
[v12 15/20] ۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg
[v12 16/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg
[v12 17/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg
[v12 18/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg
[v12 19/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۲۰.jpg
[v12 20/20] ۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg
Saved v12 outputs:
- notebook_outputs/selected_samples_v12_1_hotfix/document_summary.csv
- notebook_outputs/selected_samples_v12_1_hotfix/extracted_parts.csv
- notebook_outputs/selected_samples_v12_1_hotfix/document_json_rows.csv
- not

,count
extraction_status,
extracted_good,8
ocr_unreliable_needs_reupload,7
extracted_requires_context,5



Recommended action counts:


,count
recommended_action,
reupload_recommended,7
save_candidate,7
manual_review,6



Recommended save:


,count
recommended_save,
False,13
True,7



Page roles:


,count
page_role,
continuation_or_body_only_page,11
standalone_report_page,7
culture_result_page,1
unknown_page_role,1


,index,filename,source_relative_path,is_image,page_role,template_type,requires_backend_context_for_save,extraction_status,recommended_action,recommended_save,...,lab_result_count,patient_name_found,date_found,age_found,sex_found,national_id_hash_found,explicit_doctor_label,text_relative_path,json_relative_path,source_preview
0,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,True,continuation_or_body_only_page,thermal_or_receipt_cbc,True,extracted_requires_context,manual_review,False,...,61,False,False,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
1,2,-2147483648_-210195.jpg,samples/selected samples/-2147483648_-210195.jpg,True,continuation_or_body_only_page,emrooz_clean_screenshot,True,ocr_unreliable_needs_reupload,reupload_recommended,False,...,1,False,True,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
2,3,0014161672_14041209_O_00404121721.pdf,samples/selected samples/0014161672_14041209_O...,False,standalone_report_page,tav_text_pdf,False,extracted_good,save_candidate,True,...,19,True,True,True,True,True,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
3,4,0020139871_14041209_O_00404121714.pdf,samples/selected samples/0020139871_14041209_O...,False,standalone_report_page,tav_text_pdf,False,extracted_good,save_candidate,True,...,3,True,True,True,True,True,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
4,5,0021858845_14041209_O_00404121731.pdf,samples/selected samples/0021858845_14041209_O...,False,standalone_report_page,tav_text_pdf,False,extracted_good,save_candidate,True,...,15,True,True,True,True,True,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
5,6,0023471026_14041209_O_00404121728.pdf,samples/selected samples/0023471026_14041209_O...,False,standalone_report_page,tav_text_pdf,False,extracted_good,save_candidate,True,...,1,True,True,True,True,True,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
6,7,0024150010_14041209_O_00404121722.pdf,samples/selected samples/0024150010_14041209_O...,False,standalone_report_page,tav_text_pdf,False,extracted_good,save_candidate,True,...,19,True,True,True,True,True,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
7,8,0025631314_14041209_O_00404121726.pdf,samples/selected samples/0025631314_14041209_O...,False,standalone_report_page,tav_text_pdf,False,extracted_good,save_candidate,True,...,3,True,True,True,True,True,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
8,9,0025692283_14041209_O_00404121730.pdf,samples/selected samples/0025692283_14041209_O...,False,standalone_report_page,tav_text_pdf,False,extracted_good,save_candidate,True,...,3,True,True,True,True,True,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...
9,10,20260427_181554.jpg,samples/selected samples/20260427_181554.jpg,True,continuation_or_body_only_page,generic_lab,True,ocr_unreliable_needs_reupload,reupload_recommended,False,...,9,False,False,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hotfix...,notebook_outputs/selected_samples_v12_1_hot

,part_type,section,field_name,value,confidence,source_text,field_validation_status,field_backend_usable,index,filename,...,backend_reason_codes,corrected_numeric,corrected_value,column_statuses_json,test_name_status,result_status,unit_status,reference_range_status,flag_status,method_status
0,common_field,header,center_name,None,0.00,NaN,missing_optional,False,1,-2147483648_-210193.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,common_field,header,national_id,None,0.00,NaN,missing,False,1,-2147483648_-210193.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,common_field,header,tracking_number,None,0.00,NaN,missing_optional,False,1,-2147483648_-210193.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,common_field,header,date_of_test_or_report,None,0.00,NaN,missing_required,False,1,-2147483648_-210193.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,common_field,header,patient_name,None,0.00,NaN,missing,False,1,-2147483648_-210193.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,lab_result,urine_or_microbiology,Crystals,Not seen,0.82,Crystals\nNot seen,NaN,NaN,18,۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg,...,document_not_recommended_for_save,NaN,NaN,"{""test_name_status"": ""valid"", ""result_status"":...",valid,valid,not_applicable,missing_optional,not_applicable,missing_optional
346,lab_result,urine_or_microbiology,Urobilinogen,Normal,0.82,Urobilinogen\nNormal,NaN,NaN,18,۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg,...,document_not_recommended_for_save,NaN,NaN,"{""test_name_status"": ""valid"", ""result_status"":...",valid,valid,not_applicable,missing_optional,not_applicable,missing_optional
347,lab_result,urine_or_microbiology,Ketone,Negative,0.82,Ketone\nNegative,NaN,NaN,18,۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg,...,document_not_recommended_for_save,NaN,NaN,"{""test_name_status"": ""valid"", ""result_status"":...",valid,valid,not_applicable,missing_optional,not_applicable,missing_optional
348,lab_result,urine_or_microbiology,Nitrite,Negative,0.82,Nitrite\nNegative,NaN,NaN,18,۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg,...,document_not_recommended_for_save,NaN,NaN,"{""test_name_status"": ""valid"", ""result_status"":...",valid,valid,not_applicable,missing_optional,not_applicable,missing_optional


In [17]:

# =========================
# v12 HTML Review Report: extraction + backend recommendation
# =========================

def df_to_html_table(df, columns=None, max_rows=220):
    if df is None or df.empty:
        return "<p><em>No rows.</em></p>"
    x = df.copy()
    if columns:
        x = x[[c for c in columns if c in x.columns]]
    if len(x) > max_rows:
        x = x.head(max_rows)
    return x.to_html(index=False, escape=True)

sections_html = []
for d in documents:
    idx = d["document"]["index"]
    fn = d["document"]["filename"]
    sp = d["document"].get("source_preview")
    rec = d["persistence_recommendation"]
    page_ctx = d["document"]["page_context"]
    preview_html = f'<img src="../../{html.escape(sp)}" style="max-width:760px;border:1px solid #ddd"/>' if sp else ""
    bg = "#e8f5e9" if rec["recommended_save"] else ("#fff8e1" if rec["recommended_action"] == "manual_review" else "#ffebee")

    cand_sub = candidates_df[candidates_df["index"] == idx]
    if "score_v12_selection_score" in cand_sub.columns:
        cand_sub = cand_sub.sort_values("score_v12_selection_score", ascending=False).head(12)
    else:
        cand_sub = cand_sub.head(12)

    line_sub = lines_df[lines_df["index"] == idx].head(260)
    psub = parts_df[parts_df["index"] == idx].copy()
    common = psub[psub["part_type"] == "common_field"]
    labs = psub[psub["part_type"] == "lab_result"]
    text = d.get("extracted_text", "")

    sections_html.append(f"""
    <section style="background:{bg}; padding:16px; margin:20px 0; border-radius:8px;">
      <h2>{idx}. {html.escape(fn)}</h2>
      <p>
        <b>Extraction status:</b> {html.escape(d['document']['extraction_status'])} |
        <b>Recommended action:</b> {html.escape(rec['recommended_action'])} |
        <b>Recommended save:</b> {rec['recommended_save']} |
        <b>Reasons:</b> {html.escape(', '.join(rec['reason_codes']))}
      </p>
      <p>
        <b>Page role:</b> {html.escape(page_ctx['page_role'])} |
        <b>Template:</b> {html.escape(page_ctx['template_type'])} |
        <b>Requires backend context:</b> {page_ctx['requires_backend_context_for_save']}
      </p>
      <p>
        <b>Chosen OCR:</b> {html.escape(str(d['ocr']['variant_name']))},
        PSM={html.escape(str(d['ocr']['psm']))},
        lang={html.escape(str(d['ocr']['lang']))} |
        <b>v12 score:</b> {html.escape(str(d['ocr']['score_details'].get('v12_selection_score')))} |
        <b>OCR status:</b> {html.escape(str(d['ocr']['layout_status']))}
      </p>
      <details open><summary><b>Source preview</b></summary>{preview_html}</details>
      <details open><summary><b>Common fields</b></summary>
      {df_to_html_table(common, ['field_name','value','confidence','field_validation_status','field_backend_usable','source_text'])}
      </details>
      <details open><summary><b>Lab rows with column statuses</b></summary>
      {df_to_html_table(labs, ['field_name','value','unit','reference_range','flag','confidence','extraction_mode','row_validation_status','row_save_allowed','backend_row_save_recommendation','row_reason_codes','backend_reason_codes','test_name_status','result_status','unit_status','reference_range_status','flag_status','method_status','source_text'], 320)}
      </details>
      <details open><summary><b>Top OCR candidates</b></summary>
      {df_to_html_table(cand_sub, ['candidate_id','variant_name','psm','lang','confidence','layout_score_v9','score_v12_selection_score','layout_status','text_length','score_v12_page_role','score_v12_template_type','score_v12_extracted_row_count','score_v12_safe_row_count','is_chosen'], 16)}
      </details>
      <details><summary><b>Chosen visual lines</b></summary>
      {df_to_html_table(line_sub, ['line_id','section','text','avg_conf','left','top','width','height'], 260)}
      </details>
      <details><summary><b>Chosen OCR text</b></summary>
      <pre style="white-space:pre-wrap; background:white; padding:10px; border:1px solid #ddd;">{html.escape(text[:16000])}</pre>
      </details>
    </section>
    """)

html_parts = [
    "<!doctype html><html><head><meta charset='utf-8'/>",
    "<title>Medical Best Extraction Review v12</title>",
    "<style>",
    "body { font-family: Arial, sans-serif; margin: 24px; line-height: 1.4; }",
    "table { border-collapse: collapse; font-size: 13px; width: 100%; background:white; }",
    "td, th { border: 1px solid #ddd; padding: 6px; vertical-align: top; }",
    "th { background: #f2f2f2; }",
    "pre { direction: ltr; }",
    "</style></head><body>",
    "<h1>Medical Best Extraction Review v12</h1>",
    "<p>The API-style output returns results, statuses, scores, and evidence. Backend decides persistence.</p>",
    "<h2>Summary</h2>",
    summary_df.to_html(index=False, escape=True),
    *sections_html,
    "</body></html>",
]
review_path = OUTPUT_DIR / "review_best_extraction.html"
review_path.write_text("\n".join(html_parts), encoding="utf-8")
print("Saved:", relpath(review_path))


Saved: notebook_outputs/selected_samples_v12_1_hotfix/review_best_extraction.html


In [18]:

# =========================
# v12 QA views
# =========================

print("Documents requiring manual review or reupload:")
display(summary_df[summary_df["recommended_save"] == False][[
    "index", "filename", "is_image", "page_role", "template_type", "extraction_status",
    "recommended_action", "reason_codes", "ocr_layout_status", "ocr_v12_selection_score",
    "lab_result_count", "safe_row_count", "backend_safe_row_count",
    "patient_name_found", "date_found", "age_found", "sex_found", "json_relative_path"
]])

print("\nRows with internal valid status but blocked by document-level recommendation:")
if not parts_df.empty and "backend_row_save_recommendation" in parts_df.columns:
    blocked = parts_df[
        (parts_df["part_type"] == "lab_result") &
        (parts_df["row_save_allowed"] == True) &
        (parts_df["backend_row_save_recommendation"] == False)
    ]
    display(blocked[[
        "index", "filename", "page_role", "field_name", "value", "unit", "reference_range",
        "confidence", "row_validation_status", "backend_reason_codes", "source_text"
    ]].head(300))

print("\nAll backend save candidate rows:")
if not parts_df.empty and "backend_row_save_recommendation" in parts_df.columns:
    safe = parts_df[
        (parts_df["part_type"] == "lab_result") &
        (parts_df["backend_row_save_recommendation"] == True)
    ]
    display(safe[[
        "index", "filename", "field_name", "value", "unit", "reference_range", "flag",
        "confidence", "extraction_mode", "row_validation_status",
        "test_name_status", "result_status", "unit_status", "reference_range_status",
        "source_text"
    ]].head(500))

print("\nColumn status summary:")
if not parts_df.empty and "column_statuses_json" in parts_df.columns:
    status_counter = Counter()
    for x in parts_df.loc[parts_df["part_type"] == "lab_result", "column_statuses_json"].dropna():
        try:
            d = json.loads(x)
            for k, v in d.items():
                status_counter[f"{k}:{v}"] += 1
        except Exception:
            pass
    display(pd.DataFrame([{"status": k, "count": v} for k, v in status_counter.items()]).sort_values("count", ascending=False))


Documents requiring manual review or reupload:


,index,filename,is_image,page_role,template_type,extraction_status,recommended_action,reason_codes,ocr_layout_status,ocr_v12_selection_score,lab_result_count,safe_row_count,backend_safe_row_count,patient_name_found,date_found,age_found,sex_found,json_relative_path
0,1,-2147483648_-210193.jpg,True,continuation_or_body_only_page,thermal_or_receipt_cbc,extracted_requires_context,manual_review,extracted_requires_context|missing_or_invalid_...,usable_short_culture_text,1.000,61,20,0,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...
1,2,-2147483648_-210195.jpg,True,continuation_or_body_only_page,emrooz_clean_screenshot,ocr_unreliable_needs_reupload,reupload_recommended,missing_or_invalid_patient_identity_or_backend...,gibberish_or_bad_layout_text,0.258,1,0,0,False,True,False,False,notebook_outputs/selected_samples_v12_1_hotfix...
9,10,20260427_181554.jpg,True,continuation_or_body_only_page,generic_lab,ocr_unreliable_needs_reupload,reupload_recommended,missing_or_invalid_patient_identity_or_backend...,gibberish_or_bad_layout_text,0.439,9,0,0,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...
10,11,20260427_181636.jpg,True,continuation_or_body_only_page,atabay_rotated_or_screenshot,ocr_unreliable_needs_reupload,reupload_recommended,missing_or_invalid_patient_identity_or_backend...,gibberish_or_bad_layout_text,0.244,0,0,0,True,False,True,True,notebook_outputs/selected_samples_v12_1_hotfix...
11,12,20260427_181654.jpg,True,continuation_or_body_only_page,atabay_rotated_or_screenshot,ocr_unreliable_needs_reupload,reupload_recommended,missing_or_invalid_patient_identity_or_backend...,gibberish_or_bad_layout_text,0.342,8,0,0,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...
12,13,20260427_181713.jpg,True,culture_result_page,generic_lab,extracted_good,manual_review,missing_or_invalid_report_date,usable_short_culture_text,0.592,1,1,0,True,False,False,True,notebook_outputs/selected_samples_v12_1_hotfix...
13,14,20260427_181919.jpg,True,continuation_or_body_only_page,generic_lab,ocr_unreliable_needs_reupload,reupload_recommended,missing_or_invalid_patient_identity_or_backend...,gibberish_or_bad_layout_text,0.547,5,0,0,False,False,True,False,notebook_outputs/selected_samples_v12_1_hotfix...
14,15,۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg,True,continuation_or_body_only_page,nobin_photo_with_history_charts,extracted_requires_context,manual_review,extracted_requires_context|missing_or_invalid_...,good_layout_text,0.995,17,6,0,True,False,False,True,notebook_outputs/selected_samples_v12_1_hotfix...
15,16,۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg,True,continuation_or_body_only_page,vahdat_multi_page_photo,extracted_requires_context,manual_review,extracted_requires_context|missing_or_invalid_...,good_layout_text,1.000,26,11,0,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...
16,17,۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg,True,unknown_page_role,tav_text_pdf,ocr_unreliable_needs_reupload,reupload_recommended,missing_or_invalid_patient_identity_or_backend...,gibberish_or_bad_layout_text,0.173,0,0,0,False,False,False,False,notebook_outputs/selected_samples_v12_1_hotfix...



Rows with internal valid status but blocked by document-level recommendation:


,index,filename,page_role,field_name,value,unit,reference_range,confidence,row_validation_status,backend_reason_codes,source_text
8,1,-2147483648_-210193.jpg,continuation_or_body_only_page,EAG,7,mg/dL,NaN,0.98,valid,document_not_recommended_for_save,Estimated Average Glucose (EAG) 7 mg/dL
16,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Estradiol,250.9,pg/ml,NaN,0.98,valid,document_not_recommended_for_save,Estradiol 250.9 pg/ml CLIA 5 ی‎
36,1,-2147483648_-210193.jpg,continuation_or_body_only_page,pH,6,NaN,NaN,0.82,valid,document_not_recommended_for_save,PH 6 Bacteria Negative\nProtein Negative Mucus...
37,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Bilirubin Urine,Negative,NaN,NaN,0.82,valid,document_not_recommended_for_save,Bilirubin Negative
38,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Urobilinogen,Normal,NaN,NaN,0.82,valid,document_not_recommended_for_save,Urobilinogen Normal
40,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Nitrite,1,NaN,NaN,0.82,valid,document_not_recommended_for_save,Nitrite Negative\nBacteriology 1\nTest Result\...
55,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Urine Culture,No growth after 24 hours,NaN,NaN,0.95,valid,document_not_recommended_for_save,ite Negative Bacteriology 1 Test Result Urine ...
56,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Color,Yellow,NaN,NaN,0.76,valid,document_not_recommended_for_save,ng/mt فلت‎ 10-291 ماع‎ Urine Analysis Macrosc...
57,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Appearance,Clear,NaN,NaN,0.76,valid,document_not_recommended_for_save,ع‎ Urine Analysis Macroscopic Micrascopic Urin...
58,1,-2147483648_-210193.jpg,continuation_or_body_only_page,Protein,Negative,NaN,NaN,0.76,valid,document_not_recommended_for_save,Clear RBC. 01 Specific Gravity 1015 Epithelial...



All backend save candidate rows:


,index,filename,field_name,value,unit,reference_range,flag,confidence,extraction_mode,row_validation_status,test_name_status,result_status,unit_status,reference_range_status,source_text
86,3,0014161672_14041209_O_00404121721.pdf,WBC,7.24,10^3/uL,3.5-10.5,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,W.B.C\n7.24\n10^3/uL\n3.5-10.5
87,3,0014161672_14041209_O_00404121721.pdf,RBC,4.9,10^6/uL,4.32-5.72,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,R.B.C\n4.9\n10^6/uL\n4.32 - 5.72
88,3,0014161672_14041209_O_00404121721.pdf,HGB,15.0,g/dL,13.5-17.5,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,Hemoglobin\n15.0\ng/dL\n13.5 - 17.5
89,3,0014161672_14041209_O_00404121721.pdf,HCT,45.0,%,38.8-50,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,H.C.T\n45.0\n%\n38.8-50 %
90,3,0014161672_14041209_O_00404121721.pdf,MCV,91.8,fL,81.2-95.1,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,M.C.V\n91.8\nfL\n81.2-95.1 fl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,8,0025631314_14041209_O_00404121726.pdf,AST,18,U/L,<37,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,SGOT(AST)\n18\nU/L\n<37\nPhotometr
185,8,0025631314_14041209_O_00404121726.pdf,ALT,14,U/L,<40,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,SGPT(ALT)\n14\nU/L\n< 40\nPhotometr\nThis answ...
194,9,0025692283_14041209_O_00404121730.pdf,FBS,95,mg/dL,70-115,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,Fasting blood sugar\n95\nmg/dL\n70-115\nPhotometr
195,9,0025692283_14041209_O_00404121730.pdf,AST,14,U/L,<31,NaN,0.82,strict_multiline_or_window,valid,valid,valid,valid,valid,SGOT(AST)\n14\nU/L\n<31\nPhotometr



Column status summary:


,status,count
5,method_status:missing_optional,203
4,flag_status:not_applicable,199
0,test_name_status:valid,195
1,result_status:valid,180
3,reference_range_status:missing_optional,123
11,reference_range_status:valid,93
12,unit_status:valid,78
2,unit_status:not_applicable,72
6,unit_status:missing_required,37
15,result_status:unsafe_ocr_context,32
